# 📊 Options Strategy Analyzer

## Notebook Version 2.5 - Mixed-Regime Tightening

### Changelog
**v2.5** (Current)
- ✅ Tightened regime-based signal filtering after v2.4 validation
- ✅ Filters **Clean uptrend** CALL setups during **MIXED** regimes by default
- ✅ Preserves selective CALL setups in non-bullish conditions
- ✅ Added benchmark warm-up history in backtests to reduce early `UNKNOWN` regime classifications
- ✅ Expanded diagnostics to show **setup performance by market regime**
- ✅ Focus shifted from broad regime awareness to **transitional-market trade selection**
- ✅ Goal: reduce avoidable CALL losses in hostile or choppy environments without harming bullish edge

**v2.4**
- ✅ Added market regime filter using benchmark trend structure
- ✅ Classified market environment as BULLISH, MIXED, or BEARISH
- ✅ Restricted most bullish CALL entries during BEARISH market regimes
- ✅ Added bearish-regime PUT expansion for stronger downside participation
- ✅ Added backtest diagnostics by year, symbol, setup, exit reason, and regime
- ✅ Preserved v2.3 backtest realism improvements including non-overlapping trades
- ✅ Focus shifted from pure signal generation to regime-aware trade selection

**v2.3**
- ✅ Expanded default backtest window: 2023-01-01 → 2018-01-01
- ✅ Improved backtest realism by preventing overlapping trades on the same ticker
- ✅ Added final version review cell for research notes and lessons learned
- ✅ Fixed position sizing display to reflect current exit parameters dynamically
- ✅ Fixed visualization function typo and aligned RSI chart levels with config
- ✅ Clarified notebook version vs cell version labeling for future development
- ✅ Focus shifted toward regime validation across bullish, bearish, and mixed conditions

**v2.2**
- ✅ Optimized exit parameters through systematic testing
- ✅ Hold period reduced: 21 days → 17 days (less theta decay)
- ✅ Stop loss tightened: -50% → -35% (faster loss cuts)
- ✅ Profit target adjusted: +100% → +75% (more realistic exits)
- ✅ Performance improved through exit efficiency

**v2.1**
- ✅ Fixed PUT signal logic based on backtest analysis
- ✅ Removed 5 underperforming PUT signal conditions
- ✅ Restricted PUTs to HIGH conviction reversals in UPTREND only
- ✅ Restricted overbought PUTs to confirmed DOWNTREND only
- ✅ Win rate improved materially
- ✅ Profit factor improved materially
- ✅ Strategy rating upgraded from GOOD to EXCELLENT

**v2.0**
- ✅ Added historical backtesting framework
- ✅ Walk-forward analysis through past data
- ✅ Performance metrics: Win rate, avg return, max drawdown
- ✅ Trade-by-trade results export
- ✅ Configurable backtest period and parameters
- ✅ Simulates option P&L based on historical moves
- ✅ Separate backtest mode vs live scanning mode

**v1.8**
- ✅ Added weekly timeframe confirmation
- ✅ Multi-timeframe trend alignment check (daily + weekly)
- ✅ Enhanced signal conviction with timeframe confluence
- ✅ Filters out counter-trend trades

**v1.7**
- ✅ Added price chart visualization with signal markers
- ✅ Configurable timezone support (default: US/Eastern)
- ✅ Fixed timestamp to show correct local time in journal

**v1.6**
- ✅ Added CSV trade journal export functionality
- ✅ Automatic logging of all scanner outputs with timestamps

**v1.5**
- ✅ Fixed volume distribution/accumulation logic

**v1.4.1**
- ✅ Fixed Greeks calculation using Black-Scholes model

**v1.4**
- ✅ Added Greeks display (Delta, Gamma, Theta, Vega)

**v1.3**
- ✅ Added minimum volume and open interest filters

**v1.2**
- ✅ Added comprehensive data validation for options chains

**v1.1**
- ✅ Enhanced RSI divergence detection with proper pivot point identification

**v1.0** (Initial Release)
- Initial scanner framework with basic reversal detection

---

## Overview

### Purpose
The Options Strategy Analyzer identifies high-probability directional options trades by combining technical analysis, reversal pattern detection, market regime awareness, and systematic risk management. Strategies are validated through historical backtesting before deployment and improved version by version based on observed results.

### Core Features
- **Multi-Factor Reversal Detection**: RSI divergence, volume analysis, climax patterns, failed breakouts, MA breakdowns
- **Dynamic Position Sizing**: Adjusts contract allocation based on signal strength and enforces strict risk limits
- **Options Chain Analysis**: Selects appropriate expiration dates (45+ DTE) and near-ATM strikes with liquidity filters
- **Historical Backtesting**: Tests strategies over multiple market regimes before risking capital
- **Multi-Timeframe Analysis**: Daily + weekly trend confirmation for higher conviction
- **Market Regime Filter**: Uses benchmark trend structure to reduce bullish bias in bearish and mixed environments
- **Greeks Integration**: Complete risk profile with Delta, Theta, Vega, Gamma
- **Trade Journaling**: Automatic CSV logging of all signals and trades
- **Version Review Framework**: Captures the Good, the Bad, the Ugly, and future ideas after each major version

### Performance Status (v2.5)
**Current status**: Validation in progress

v2.5 is focused on tightening trade selection in **MIXED** regimes after v2.4 showed that many weak CALL losses were still occurring in transitional conditions rather than only in clearly bearish ones.

### Workflow
1. Configure scanner parameters (tickers, thresholds, account size)
2. Determine broader market regime using benchmark trend structure
3. Run backtest over multiple market conditions
4. Review diagnostics by year, symbol, setup, exit reason, and regime
5. Tighten or relax trade filters based on evidence
6. Switch to live mode for real-time signal generation
7. Review options chain, Greeks, and position sizing
8. Track strategy evolution through the version review cell

### Recent Improvements (v2.5)
**Mixed-Regime Tightening**: v2.4 showed that the strategy improved only modestly because many weak long-call losses were still happening in **MIXED** market regimes. This version specifically targets that problem by filtering out **Clean uptrend** CALL entries in mixed conditions by default, while preserving more selective oversold and bullish-reversal setups. Backtests also now load extra benchmark warm-up history so market regime labels are available earlier in the test window.

In [1]:
# Code Cell 1: Library Imports (v1.7 - Added matplotlib & timezone)

import yfinance as yf

for symbol in ["AAPL", "MSFT", "GOOGL"]:
    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="max")

    if hist.empty:
        print(f"{symbol}: No data returned")
    else:
        print(f"{symbol}:")
        print(f"  First date: {hist.index.min().date()}")
        print(f"  Last date : {hist.index.max().date()}")
        print(f"  Rows      : {len(hist)}")
        print()

import pandas as pd
import numpy as np
from datetime import datetime
from scipy.stats import norm
import warnings
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pytz import timezone as pytz_timezone

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*timezone.*')

# Configure pandas display for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✓ Libraries loaded successfully (v1.7)")

AAPL:
  First date: 1980-12-12
  Last date : 2026-05-15
  Rows      : 11448

MSFT:
  First date: 1986-03-13
  Last date : 2026-05-15
  Rows      : 10122

GOOGL:
  First date: 2004-08-19
  Last date : 2026-05-15
  Rows      : 5470

✓ Libraries loaded successfully (v1.7)


In [2]:
# Code Cell 1A: Interactive Configuration
# Notebook Version: v2.5
# Cell Version: v2.1

print("="*62)
print(" INTERACTIVE CONFIGURATION")
print("="*62)

# ──────────────────────────────────────────────────────────────
# TICKERS
# ──────────────────────────────────────────────────────────────
print("\nEnter ticker symbols separated by commas (e.g., GOOGL, AAPL, MSFT)")
ticker_input = input("Tickers: ").strip()

if ticker_input:
    TICKERS = [t.strip().upper() for t in ticker_input.split(",")]
    print(f"✓ Tickers set to: {TICKERS}")
else:
    TICKERS = ["GOOGL", "AAPL"]
    print(f"✓ Using default tickers: {TICKERS}")

# ──────────────────────────────────────────────────────────────
# MODE SELECTION
# ──────────────────────────────────────────────────────────────
print("\nSelect mode:")
print("  1 = Live Scan (real-time signals)")
print("  2 = Backtest (historical analysis)")
mode_input = input("Mode (1 or 2): ").strip()

if mode_input == "2":
    BACKTEST_MODE = True
    print("✓ BACKTEST MODE enabled")
else:
    BACKTEST_MODE = False
    print("✓ LIVE SCAN MODE enabled")

# ──────────────────────────────────────────────────────────────
# FEATURE TOGGLES
# ──────────────────────────────────────────────────────────────
print("\nEnable charts? (y/n)")
charts_input = input("Charts: ").strip().lower()
ENABLE_CHARTS = (charts_input == "y")
print(f"✓ Charts: {'Enabled' if ENABLE_CHARTS else 'Disabled'}")

print("\nEnable trade journal? (y/n)")
journal_input = input("Journal: ").strip().lower()
ENABLE_JOURNAL = (journal_input == "y")
print(f"✓ Journal: {'Enabled' if ENABLE_JOURNAL else 'Disabled'}")

# ──────────────────────────────────────────────────────────────
# BACKTEST SETTINGS
# ──────────────────────────────────────────────────────────────
if BACKTEST_MODE:
    print("\nBacktest start date (YYYY-MM-DD) - press Enter for default (2018-01-01)")
    start_input = input("Start date: ").strip()
    BACKTEST_START_DATE = start_input if start_input else "2018-01-01"

    print("\nBacktest end date (YYYY-MM-DD) - press Enter for default (2024-12-31)")
    end_input = input("End date: ").strip()
    BACKTEST_END_DATE = end_input if end_input else "2024-12-31"

    print(f"✓ Backtest period: {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")
else:
    BACKTEST_START_DATE = "2018-01-01"
    BACKTEST_END_DATE = "2024-12-31"

# ──────────────────────────────────────────────────────────────
# SUMMARY
# ──────────────────────────────────────────────────────────────
print("\n" + "="*62)
print(" CONFIGURATION SUMMARY")
print("="*62)
print(f" Tickers      : {', '.join(TICKERS)}")
print(f" Mode         : {'BACKTEST' if BACKTEST_MODE else 'LIVE SCAN'}")
print(f" Charts       : {'ON' if ENABLE_CHARTS else 'OFF'}")
print(f" Journal      : {'ON' if ENABLE_JOURNAL else 'OFF'}")
if BACKTEST_MODE:
    print(f" Backtest     : {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")
print("="*62)
print("\n✓ Interactive configuration complete")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.1")

 INTERACTIVE CONFIGURATION

Enter ticker symbols separated by commas (e.g., GOOGL, AAPL, MSFT)
Tickers: GOOGL, AAPL, MSFT
✓ Tickers set to: ['GOOGL', 'AAPL', 'MSFT']

Select mode:
  1 = Live Scan (real-time signals)
  2 = Backtest (historical analysis)
Mode (1 or 2): 2
✓ BACKTEST MODE enabled

Enable charts? (y/n)
Charts: n
✓ Charts: Disabled

Enable trade journal? (y/n)
Journal: n
✓ Journal: Disabled

Backtest start date (YYYY-MM-DD) - press Enter for default (2018-01-01)
Start date: 

Backtest end date (YYYY-MM-DD) - press Enter for default (2024-12-31)
End date: 
✓ Backtest period: 2018-01-01 to 2024-12-31

 CONFIGURATION SUMMARY
 Tickers      : GOOGL, AAPL, MSFT
 Mode         : BACKTEST
 Charts       : OFF
 Journal      : OFF
 Backtest     : 2018-01-01 to 2024-12-31

✓ Interactive configuration complete
  Notebook Version: v2.5
  Cell Version: v2.1


## Text Cell 2: Notebook Configuration

### Scanner Parameters
- **Tickers**: Symbols to scan (update this list for your watchlist or backtest basket)
- **Options Settings**: Minimum DTE (45 days), IV percentile threshold (50th)
- **Technical Levels**: RSI oversold/overbought thresholds, moving average periods

### Liquidity Filters (New in v1.3)
- **Minimum Volume**: 10 contracts traded today (prevents stale options)
- **Minimum Open Interest**: 50 contracts (ensures you can exit when needed)
- **Spread Warning**: Alert if bid/ask spread exceeds 20%
- **ATM Tolerance**: Only consider strikes within ±10% of current price

### Risk Management
- **Account Size**: Total capital allocated to options trading ($4,000 default)
- **Risk Per Trade**: 3-5% of account, adjusted by signal strength
- **Position Limits**: Max 4 open positions, 20% max allocation per trade
- **Exit Rules**: 35% stop loss, 75% profit target
- **Backtest Hold Period**: 17 trading days maximum hold per position

### Market Regime Filter
- **Benchmark Symbol**: Default uses `SPY` to classify overall market regime
- **Regime Logic**: Price vs MA50 and MA200
- **Bullish Regime**: Price > MA50 > MA200
- **Bearish Regime**: Price < MA50 < MA200
- **Mixed Regime**: Anything in between
- **v2.5 Tightening**: `Clean uptrend` CALL setups are filtered in **MIXED** regimes by default
- **Warm-Up History**: Benchmark data is loaded before the official backtest start date so regime labels are available earlier

### Reversal Detection Tuning
Fine-tune sensitivity for pattern recognition:
- **Divergence Lookback**: 14 bars for RSI pivot comparison
- **Volume Analysis**: 20-bar average baseline, 2x volume = climax
- **Extension Threshold**: 7% above MA20 = overextended
- **Breakout Validation**: 10-bar window for failed breakout detection

### Backtest Scope
- **Default Backtest Start**: 2018-01-01
- **Purpose**: Reduce regime bias by including bullish, bearish, and mixed market conditions
- **Research Goal**: Validate whether strategy performance holds outside the 2023-2024 large-cap tech environment

In [20]:
# Code Cell 2: Configuration Constants
# Notebook Version: v2.5
# Cell Version: v2.5

# ─────────────────────────────────────────────────────────────────────
# SCANNER CONFIGURATION (Set by Code Cell 1A)
# ─────────────────────────────────────────────────────────────────────

if 'TICKERS' not in dir():
    TICKERS = ["GOOGL", "AAPL"]
if 'BACKTEST_MODE' not in dir():
    BACKTEST_MODE = False
if 'ENABLE_CHARTS' not in dir():
    ENABLE_CHARTS = True
if 'ENABLE_JOURNAL' not in dir():
    ENABLE_JOURNAL = True
if 'BACKTEST_START_DATE' not in dir():
    BACKTEST_START_DATE = "2018-01-01"
if 'BACKTEST_END_DATE' not in dir():
    BACKTEST_END_DATE = "2024-12-31"

# Options parameters
MIN_DTE = 45
IV_PCT_THRESHOLD = 50

# Technical indicator settings
RSI_OVERSOLD = 35
RSI_OVERBOUGHT = 65
MA_SHORT = 20
MA_LONG = 50

# ─────────────────────────────────────────────────────────────────────
# MULTI-TIMEFRAME ANALYSIS
# ─────────────────────────────────────────────────────────────────────

ENABLE_MTF = True
REQUIRE_MTF_ALIGNMENT = True
WEEKLY_MA_SHORT = 10
WEEKLY_MA_LONG = 20

# ─────────────────────────────────────────────────────────────────────
# MARKET REGIME FILTER
# ─────────────────────────────────────────────────────────────────────

ENABLE_REGIME_FILTER = True
REGIME_SYMBOL = "SPY"                 # Can test QQQ later if desired
REGIME_MA_SHORT = 50
REGIME_MA_LONG = 200
REGIME_WARMUP_DAYS = 400              # Extra benchmark history before backtest start

ALLOW_MIXED_CLEAN_UPTREND_CALLS = False
ALLOW_BEARISH_COUNTERTREND_CALLS = True
BEARISH_COUNTERTREND_MAX_RSI = 45

# ─────────────────────────────────────────────────────────────────────
# LIQUIDITY FILTERS
# ─────────────────────────────────────────────────────────────────────

MIN_OPTION_VOLUME = 10
MIN_OPTION_OI = 50
MAX_SPREAD_PCT = 20
ATM_STRIKE_TOLERANCE = 0.10

# ─────────────────────────────────────────────────────────────────────
# RISK MANAGEMENT
# ─────────────────────────────────────────────────────────────────────

ACCOUNT_SIZE = 4000.00
RISK_PCT_MIN = 0.03
RISK_PCT_MAX = 0.05
MAX_POSITIONS = 4
STOP_LOSS_PCT = 0.35
PROFIT_TARGET_PCT = 0.75

# ─────────────────────────────────────────────────────────────────────
# REVERSAL DETECTION TUNING
# ─────────────────────────────────────────────────────────────────────

DIVERGENCE_LOOKBACK = 14
VOLUME_AVG_PERIOD = 20
CLIMAX_VOLUME_MULT = 2.0
CLIMAX_MOVE_PCT = 0.03
EXTENSION_PCT = 0.07
BREAKOUT_LOOKBACK = 10

# ─────────────────────────────────────────────────────────────────────
# TRADE JOURNAL
# ─────────────────────────────────────────────────────────────────────

JOURNAL_PATH = "trade_journal.csv"

# ─────────────────────────────────────────────────────────────────────
# VISUALIZATION
# ─────────────────────────────────────────────────────────────────────

CHART_PERIOD = 90
TIMEZONE = 'US/Eastern'

# ─────────────────────────────────────────────────────────────────────
# BACKTESTING
# ─────────────────────────────────────────────────────────────────────

BACKTEST_RESULTS_PATH = "backtest_results.csv"
BACKTEST_HOLD_DAYS = 17
BACKTEST_SLIPPAGE = 0.05
BACKTEST_COMMISSION = 1.00

print("✓ Configuration loaded")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")
print(f"  Scanning: {', '.join(TICKERS)}")
print(f"  Account: ${ACCOUNT_SIZE:,.2f} | Risk per trade: {RISK_PCT_MIN*100:.0f}%-{RISK_PCT_MAX*100:.0f}%")
print(f"  Multi-Timeframe: {'Enabled' if ENABLE_MTF else 'Disabled'} | Require Alignment: {REQUIRE_MTF_ALIGNMENT}")
print(f"  Market Regime Filter: {'Enabled' if ENABLE_REGIME_FILTER else 'Disabled'} | Benchmark: {REGIME_SYMBOL}")
print(f"  Mixed Clean Uptrend CALLs Allowed: {ALLOW_MIXED_CLEAN_UPTREND_CALLS}")
print(f"  Bearish Countertrend CALLs Allowed: {ALLOW_BEARISH_COUNTERTREND_CALLS}")
print(f"  Liquidity: Min volume {MIN_OPTION_VOLUME}, Min OI {MIN_OPTION_OI}")
print(f"  Journal: {'Enabled' if ENABLE_JOURNAL else 'Disabled'} → {JOURNAL_PATH if ENABLE_JOURNAL else 'N/A'}")
print(f"  Charts: {'Enabled' if ENABLE_CHARTS else 'Disabled'} | Timezone: {TIMEZONE}")
print(f"  Backtest Mode: {'ENABLED' if BACKTEST_MODE else 'Disabled'}")
if BACKTEST_MODE:
    print(f"  Backtest Period: {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")

✓ Configuration loaded
  Notebook Version: v2.5
  Cell Version: v2.5
  Scanning: GOOGL, AAPL, MSFT
  Account: $4,000.00 | Risk per trade: 3%-5%
  Multi-Timeframe: Enabled | Require Alignment: True
  Market Regime Filter: Enabled | Benchmark: SPY
  Mixed Clean Uptrend CALLs Allowed: False
  Bearish Countertrend CALLs Allowed: True
  Liquidity: Min volume 10, Min OI 50
  Journal: Disabled → N/A
  Charts: Disabled | Timezone: US/Eastern
  Backtest Mode: ENABLED
  Backtest Period: 2018-01-01 to 2024-12-31


## Text Cell 3: Technical Indicators

### Indicator Functions
These functions calculate the core technical metrics used for signal generation:

**RSI (Relative Strength Index)**: Measures momentum on 0-100 scale. Values below 35 suggest oversold conditions (potential buy), above 65 suggest overbought (potential sell).

**Historical Volatility**: Annualized standard deviation of log returns, used as proxy for options pricing volatility when IV data is incomplete.

**IV Percentile**: Ranks current implied volatility against 1-year history. High percentile (>50th) warns that options are expensive relative to historical norms.

**Trend Classification**: Compares price to MA20 and MA50 to determine market structure (UPTREND, DOWNTREND, or MIXED).

In [3]:
# Code Cell 3: Indicator Functions

def compute_rsi(series, period=14):
    """
    Calculate RSI using standard Wilder's smoothing method.
    Returns: pandas Series with RSI values (0-100)
    """
    delta = series.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, np.nan)  # Prevent division by zero
    return 100 - (100 / (1 + rs))


def compute_historical_volatility(close, window=30):
    """
    Calculate annualized historical volatility.
    Returns: pandas Series with volatility as percentage
    """
    log_ret = np.log(close / close.shift(1))
    return log_ret.rolling(window).std() * np.sqrt(252) * 100


def compute_iv_percentile(series, current):
    """
    Rank current value against historical distribution.
    Returns: Percentile rank (0-100) or None if insufficient data
    """
    if len(series) < 20:
        return None
    return round((series < current).mean() * 100, 1)


def get_trend(price, ma_s, ma_l):
    """
    Classify trend based on price vs moving averages.
    Returns: "UPTREND", "DOWNTREND", or "MIXED"
    """
    if price > ma_s > ma_l:
        return "UPTREND"
    elif price < ma_s < ma_l:
        return "DOWNTREND"
    return "MIXED"


print("✓ Indicator functions defined")

✓ Indicator functions defined


## Text Cell 3B: Multi-Timeframe Analysis

### Purpose
Multi-timeframe analysis compares daily and weekly charts to filter out counter-trend trades. When both timeframes agree on direction, conviction increases. When they conflict, the trade is flagged or filtered out.

### How It Works
**Daily Timeframe**: Primary analysis using 20/50 moving averages
**Weekly Timeframe**: Confirmation using 10/20 week moving averages (≈50/100 day equivalents)

### Alignment Rules
**High Confidence (Boost conviction)**:
- CALL: Daily uptrend + Weekly uptrend
- PUT: Daily downtrend + Weekly downtrend

**Moderate Confidence (Accept)**:
- CALL: Daily uptrend + Weekly mixed
- PUT: Daily downtrend + Weekly mixed

**Conflict (Filter out if REQUIRE_MTF_ALIGNMENT = True)**:
- CALL signal but weekly is in downtrend
- PUT signal but weekly is in uptrend

### Benefits
- Filters out counter-trend trades (lower win rate)
- Increases conviction when timeframes align
- Reduces whipsaw trades
- Better risk/reward on aligned setups

### Configuration
- `ENABLE_MTF = True/False` - Turn multi-timeframe on/off
- `REQUIRE_MTF_ALIGNMENT = True/False` - Strict filtering vs warning only

In [4]:
# Code Cell 3B: Multi-Timeframe Functions (NEW IN v1.8)

def analyze_weekly_timeframe(ticker_obj):
    """
    Analyze weekly chart for trend confirmation.

    Returns:
        Dictionary with weekly analysis data or None if error
    """
    try:
        # Fetch weekly data (2 years for enough weekly bars)
        weekly_hist = ticker_obj.history(period="2y", interval="1wk")

        if weekly_hist.empty or len(weekly_hist) < WEEKLY_MA_LONG + 5:
            return None

        weekly_close = weekly_hist['Close']
        weekly_price = round(weekly_close.iloc[-1], 2)

        # Calculate weekly indicators
        weekly_rsi_s = compute_rsi(weekly_close)
        weekly_rsi = round(weekly_rsi_s.iloc[-1], 1)

        weekly_ma_s = weekly_close.rolling(WEEKLY_MA_SHORT).mean()
        weekly_ma_l = weekly_close.rolling(WEEKLY_MA_LONG).mean()

        weekly_ma_s_current = weekly_ma_s.iloc[-1]
        weekly_ma_l_current = weekly_ma_l.iloc[-1]

        weekly_trend = get_trend(weekly_price, weekly_ma_s_current, weekly_ma_l_current)

        weekly_p_ma_s = round((weekly_price - weekly_ma_s_current) / weekly_ma_s_current * 100, 2)
        weekly_p_ma_l = round((weekly_price - weekly_ma_l_current) / weekly_ma_l_current * 100, 2)

        return {
            'price': weekly_price,
            'rsi': weekly_rsi,
            'trend': weekly_trend,
            'ma_short': round(weekly_ma_s_current, 2),
            'ma_long': round(weekly_ma_l_current, 2),
            'pct_from_ma_short': weekly_p_ma_s,
            'pct_from_ma_long': weekly_p_ma_l
        }

    except Exception as e:
        print(f"   ⚠ Weekly timeframe analysis failed: {e}")
        return None


def check_timeframe_alignment(daily_trend, weekly_trend, signal_direction):
    """
    Check if daily and weekly timeframes are aligned with signal direction.

    Args:
        daily_trend: Daily trend classification
        weekly_trend: Weekly trend classification
        signal_direction: "BUY CALL" or "BUY PUT"

    Returns:
        (is_aligned, confidence_boost, description)
    """
    if not ENABLE_MTF:
        return True, 0, "Multi-timeframe disabled"

    # Determine if trends align
    both_up = (daily_trend == "UPTREND" and weekly_trend == "UPTREND")
    both_down = (daily_trend == "DOWNTREND" and weekly_trend == "DOWNTREND")
    daily_up_weekly_mixed = (daily_trend == "UPTREND" and weekly_trend == "MIXED")
    daily_down_weekly_mixed = (daily_trend == "DOWNTREND" and weekly_trend == "MIXED")

    # Check alignment with signal direction
    if signal_direction == "BUY CALL":
        # Bullish signal - want uptrends
        if both_up:
            return True, 1, "✓ Daily & weekly both uptrending - HIGH CONFIDENCE"
        elif daily_up_weekly_mixed:
            return True, 0, "✓ Daily uptrend, weekly mixed - MODERATE CONFIDENCE"
        elif daily_trend == "MIXED" and weekly_trend == "UPTREND":
            return True, 0, "✓ Weekly uptrend supporting - MODERATE CONFIDENCE"
        elif daily_trend == "UPTREND" and weekly_trend == "DOWNTREND":
            return False, -1, "⚠ CONFLICT: Daily up but weekly down - COUNTER-TREND TRADE"
        else:
            return False, 0, "⚠ Weak alignment - consider passing"

    else:  # BUY PUT
        # Bearish signal - want downtrends
        if both_down:
            return True, 1, "✓ Daily & weekly both downtrending - HIGH CONFIDENCE"
        elif daily_down_weekly_mixed:
            return True, 0, "✓ Daily downtrend, weekly mixed - MODERATE CONFIDENCE"
        elif daily_trend == "MIXED" and weekly_trend == "DOWNTREND":
            return True, 0, "✓ Weekly downtrend supporting - MODERATE CONFIDENCE"
        elif daily_trend == "DOWNTREND" and weekly_trend == "UPTREND":
            return False, -1, "⚠ CONFLICT: Daily down but weekly up - COUNTER-TREND TRADE"
        else:
            return False, 0, "⚠ Weak alignment - consider passing"


print("✓ Multi-timeframe functions defined (v1.8)")

✓ Multi-timeframe functions defined (v1.8)


## Text Cell 3C: Market Regime Filter

### Purpose
The market regime filter helps prevent the strategy from taking the same type of trade in every environment. A bullish trend-following setup may work well in a strong market but fail repeatedly in a transitional or deteriorating one.

### How It Works
A benchmark symbol, usually `SPY`, is analyzed using:
- **Price**
- **50-day moving average**
- **200-day moving average**

### Regime Definitions
- **BULLISH**: Price > MA50 > MA200
- **BEARISH**: Price < MA50 < MA200
- **MIXED**: Any condition between those two extremes

### v2.5 Tightening
v2.4 showed that many weak losses were still happening in **MIXED** conditions, not just clearly bearish ones.  
To address that:

- **BULLISH**: Standard CALL logic remains available
- **MIXED**: `Clean uptrend` CALL setups are filtered by default
- **MIXED**: More selective CALL setups such as oversold support and bullish reversal can still be considered
- **BEARISH**: Most CALL setups remain blocked, except for rare counter-trend bounce exceptions if enabled
- **PUTs**: Bearish participation remains selective and should improve only after regime filtering proves effective

### Why This Matters
Long options are vulnerable to both price drift and theta decay. Transitional environments often produce just enough strength to trigger entries, but not enough follow-through to support profitable long CALL trades. v2.5 is designed to become more conservative in exactly that type of market.

In [21]:
# Code Cell 3C: Market Regime Functions
# Notebook Version: v2.5
# Cell Version: v2.5

def classify_market_regime(price, ma_short, ma_long):
    """
    Classify broad market regime using benchmark price vs MA structure.
    Returns: "BULLISH", "BEARISH", "MIXED", or "UNKNOWN"
    """
    if pd.isna(price) or pd.isna(ma_short) or pd.isna(ma_long):
        return "UNKNOWN"

    if price > ma_short > ma_long:
        return "BULLISH"
    elif price < ma_short < ma_long:
        return "BEARISH"
    return "MIXED"


def get_regime_from_history(close_series, short_period=None, long_period=None):
    """
    Determine market regime from a historical close series.

    Returns:
        Dictionary with regime details or None if insufficient data
    """
    if short_period is None:
        short_period = REGIME_MA_SHORT
    if long_period is None:
        long_period = REGIME_MA_LONG

    if close_series is None:
        return None

    close_series = close_series.dropna().copy()
    if len(close_series) < long_period + 5:
        return None

    ma_short = close_series.rolling(short_period).mean()
    ma_long = close_series.rolling(long_period).mean()

    price = round(close_series.iloc[-1], 2)
    ma_short_current = ma_short.iloc[-1]
    ma_long_current = ma_long.iloc[-1]

    if pd.isna(ma_short_current) or pd.isna(ma_long_current):
        return None

    ma_short_current = round(ma_short_current, 2)
    ma_long_current = round(ma_long_current, 2)

    regime = classify_market_regime(price, ma_short_current, ma_long_current)

    return {
        'price': price,
        'ma_short': ma_short_current,
        'ma_long': ma_long_current,
        'regime': regime,
        'pct_from_ma_short': round((price - ma_short_current) / ma_short_current * 100, 2) if ma_short_current else np.nan,
        'pct_from_ma_long': round((price - ma_long_current) / ma_long_current * 100, 2) if ma_long_current else np.nan,
    }


def analyze_market_regime(symbol=REGIME_SYMBOL):
    """
    Analyze current market regime using benchmark symbol.
    Returns:
        Dictionary with regime data or None if unavailable
    """
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period="3y")

        if hist.empty or 'Close' not in hist.columns:
            return None

        regime_data = get_regime_from_history(hist['Close'])
        return regime_data

    except Exception as e:
        print(f"   ⚠ Market regime analysis failed: {e}")
        return None


def filter_signals_for_regime(signals, market_regime, trend, rsi, bullish_rev):
    """
    Filter directional signals based on broader market regime.

    v2.5 logic:
    - BULLISH: allow current signal set
    - MIXED: filter 'Clean uptrend' CALLs by default
    - BEARISH: block most CALLs, allow only selective bullish-reversal bounce setups

    Returns:
        (filtered_signals, notes)
    """
    if not ENABLE_REGIME_FILTER or not signals or market_regime in (None, "UNKNOWN"):
        return signals, []

    filtered = []
    notes = []

    for direction, reason in signals:
        allow_signal = True

        if market_regime == "MIXED" and direction == "BUY CALL":
            if "Clean uptrend" in reason and not ALLOW_MIXED_CLEAN_UPTREND_CALLS:
                allow_signal = False
                notes.append(f"Filtered {direction} in MIXED regime — {reason}")

        elif market_regime == "BEARISH" and direction == "BUY CALL":
            allow_signal = (
                ALLOW_BEARISH_COUNTERTREND_CALLS
                and "Bullish reversal" in reason
                and len(bullish_rev) > 0
                and rsi <= BEARISH_COUNTERTREND_MAX_RSI
                and trend != "DOWNTREND"
            )

            if not allow_signal:
                notes.append(f"Filtered {direction} in BEARISH regime — {reason}")

        if allow_signal:
            filtered.append((direction, reason))

    return filtered, notes


print("✓ Market regime functions defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Market regime functions defined
  Notebook Version: v2.5
  Cell Version: v2.5


## Text Cell 4A: Understanding Option Greeks

### What Are Greeks?
Greeks measure how option prices change relative to various factors. Understanding them is critical for managing risk.

**Delta (Δ)**: Rate of price change per $1 move in underlying
- Calls: 0 to 1.00 (ATM ≈ 0.50, meaning option moves $0.50 per $1 stock move)
- Puts: 0 to -1.00 (ATM ≈ -0.50, meaning option moves $0.50 per $1 stock move down)
- Higher delta = more directional exposure = closer to stock movement

**Theta (Θ)**: Daily time decay (always negative for long options)
- Shows how much option loses in value each day, all else equal
- Accelerates as expiration approaches
- Example: Theta of -$0.15 means losing $15/day per contract

**Vega (ν)**: Sensitivity to 1% change in implied volatility
- Positive for long options (benefit from IV increase)
- Example: Vega of 0.10 means +$10 per contract if IV rises 1%
- Important during earnings or events that spike IV

**Gamma (Γ)**: Rate of change in Delta
- Measures how fast Delta changes as stock moves
- Highest for ATM options
- High gamma = Delta changes rapidly (more risk/reward)

### Practical Use
- **High Theta**: Pay attention to time decay, may need quick move
- **High Vega**: Vulnerable to IV crush, avoid pre-earnings
- **Low Delta (<0.30)**: Option may not move much even if direction right
- **High Delta (>0.70)**: Acts almost like stock, less leverage

In [5]:
# Code Cell 4A: Black-Scholes Greeks Calculator (NEW IN v1.4.1)

def calculate_greeks(option_type, stock_price, strike, time_to_expiry, risk_free_rate, volatility):
    """
    Calculate option Greeks using Black-Scholes model.

    Args:
        option_type: 'call' or 'put'
        stock_price: Current stock price
        strike: Option strike price
        time_to_expiry: Time to expiration in years
        risk_free_rate: Annual risk-free rate (default 0.05 for 5%)
        volatility: Implied volatility as decimal (e.g., 0.30 for 30%)

    Returns:
        Dictionary with delta, gamma, theta, vega
    """

    # Handle edge cases
    if time_to_expiry <= 0:
        # Option expired
        if option_type == 'call':
            delta = 1.0 if stock_price > strike else 0.0
        else:
            delta = -1.0 if stock_price < strike else 0.0
        return {'delta': delta, 'gamma': 0, 'theta': 0, 'vega': 0}

    if volatility <= 0 or stock_price <= 0 or strike <= 0:
        return {'delta': np.nan, 'gamma': np.nan, 'theta': np.nan, 'vega': np.nan}

    # Black-Scholes calculations
    d1 = (np.log(stock_price / strike) + (risk_free_rate + 0.5 * volatility**2) * time_to_expiry) / (volatility * np.sqrt(time_to_expiry))
    d2 = d1 - volatility * np.sqrt(time_to_expiry)

    # Calculate Greeks
    if option_type.lower() == 'call':
        delta = norm.cdf(d1)
    else:  # put
        delta = -norm.cdf(-d1)

    # Gamma (same for calls and puts)
    gamma = norm.pdf(d1) / (stock_price * volatility * np.sqrt(time_to_expiry))

    # Theta (daily decay)
    if option_type.lower() == 'call':
        theta = (-(stock_price * norm.pdf(d1) * volatility) / (2 * np.sqrt(time_to_expiry))
                 - risk_free_rate * strike * np.exp(-risk_free_rate * time_to_expiry) * norm.cdf(d2)) / 365
    else:  # put
        theta = (-(stock_price * norm.pdf(d1) * volatility) / (2 * np.sqrt(time_to_expiry))
                 + risk_free_rate * strike * np.exp(-risk_free_rate * time_to_expiry) * norm.cdf(-d2)) / 365

    # Vega (sensitivity to 1% change in IV)
    vega = stock_price * norm.pdf(d1) * np.sqrt(time_to_expiry) / 100

    return {
        'delta': delta,
        'gamma': gamma,
        'theta': theta,
        'vega': vega
    }


def get_greeks_for_option(option_row, stock_price, time_to_expiry, option_type):
    """
    Extract or calculate Greeks for an option.

    Args:
        option_row: DataFrame row from yfinance option chain
        stock_price: Current stock price
        time_to_expiry: Years to expiration
        option_type: 'call' or 'put'

    Returns:
        Dictionary with delta, gamma, theta, vega
    """
    # Try to get Greeks from yfinance first
    delta = option_row.get('delta', np.nan)
    gamma = option_row.get('gamma', np.nan)
    theta = option_row.get('theta', np.nan)
    vega = option_row.get('vega', np.nan)

    # If any are missing, calculate them ourselves
    if np.isnan(delta) or np.isnan(gamma) or np.isnan(theta) or np.isnan(vega):
        strike = option_row.get('strike', 0)
        iv = option_row.get('impliedVolatility', 0)

        if iv > 0 and strike > 0 and time_to_expiry > 0:
            # Use 5% risk-free rate as default (adjust based on current rates)
            calculated = calculate_greeks(option_type, stock_price, strike, time_to_expiry, 0.05, iv)

            # Use calculated values if yfinance didn't provide them
            if np.isnan(delta):
                delta = calculated['delta']
            if np.isnan(gamma):
                gamma = calculated['gamma']
            if np.isnan(theta):
                theta = calculated['theta']
            if np.isnan(vega):
                vega = calculated['vega']

    return {
        'delta': delta,
        'gamma': gamma,
        'theta': theta,
        'vega': vega
    }


print("✓ Greeks calculator defined (v1.4.1)")

✓ Greeks calculator defined (v1.4.1)


## Text Cell 4: Reversal Detection Framework

### Pattern Recognition Logic
The scanner identifies potential trend reversals by detecting multiple confirming signals. No single indicator is sufficient; edge comes from confluence.

**RSI Divergence**: Price makes new high while RSI makes lower high (bearish), or price makes new low while RSI makes higher low (bullish). Suggests momentum is diverging from price action.

**Volume Signals (Corrected in v1.5)**:
- *Distribution*: Down days on heavy volume (institutions selling into strength) - **BEARISH**
- *Selling Pressure*: Multiple down days with above-average volume (sustained selling) - **BEARISH**
- *Accumulation*: Up days on heavy volume (institutions buying weakness) - **BULLISH**
- *Weak Rally*: Up days on low volume (retail buying, unsustainable) - **BEARISH**

**Why Volume Matters**: Professional traders (institutions, market makers) move size and create volume. When price rises on low volume, it suggests retail participation without institutional support - often precedes reversals. When price falls on high volume, institutions are distributing (selling) - bearish signal.

**Climax Detection**: Massive single-bar move on huge volume after extended run — often marks exhaustion before reversal.

**Failed Breakout**: Price pushes to new high then reverses below prior resistance, trapping late buyers.

**MA Breakdown**: Price closes below moving average for 2+ consecutive bars after being above, confirming loss of support.

### Conviction Scoring
Multiple confirming signals increase conviction:
- **WATCH** (2 signals): Early warning, monitor closely
- **MODERATE** (3 signals): Actionable setup, standard position size
- **HIGH** (4+ signals): High-conviction setup, maximum position size

In [10]:
# Code Cell 4: Reversal Detection Functions
# Notebook Version: v2.5
# Cell Version: v1.5

def detect_rsi_divergence(close, rsi_series, lookback=None):
    """
    Detect RSI divergence patterns using proper pivot point identification.
    Bearish: price swing high > prior swing high, RSI swing high < prior RSI high
    Bullish: price swing low < prior swing low, RSI swing low > prior RSI low
    Returns: List of detected divergence types
    """
    if lookback is None:
        lookback = DIVERGENCE_LOOKBACK

    results = []
    if len(close) < lookback + 5:
        return results

    # Find swing highs (for bearish divergence detection)
    swing_highs_price = []
    swing_highs_rsi = []

    for i in range(-lookback, -2):
        idx = len(close) + i
        if idx <= 0 or idx >= len(close) - 1:
            continue

        if close.iloc[idx] > close.iloc[idx-1] and close.iloc[idx] > close.iloc[idx+1]:
            swing_highs_price.append((i, close.iloc[idx]))
            swing_highs_rsi.append((i, rsi_series.iloc[idx]))

    # Check for bearish divergence
    if len(swing_highs_price) >= 2:
        latest_price = swing_highs_price[-1][1]
        prior_price = swing_highs_price[-2][1]
        latest_rsi = swing_highs_rsi[-1][1]
        prior_rsi = swing_highs_rsi[-2][1]

        if latest_price > prior_price and latest_rsi < prior_rsi:
            if rsi_series.iloc[-1] > 50:
                results.append("BEARISH_DIVERGENCE")

    # Find swing lows (for bullish divergence detection)
    swing_lows_price = []
    swing_lows_rsi = []

    for i in range(-lookback, -2):
        idx = len(close) + i
        if idx <= 0 or idx >= len(close) - 1:
            continue

        if close.iloc[idx] < close.iloc[idx-1] and close.iloc[idx] < close.iloc[idx+1]:
            swing_lows_price.append((i, close.iloc[idx]))
            swing_lows_rsi.append((i, rsi_series.iloc[idx]))

    # Check for bullish divergence
    if len(swing_lows_price) >= 2:
        latest_price = swing_lows_price[-1][1]
        prior_price = swing_lows_price[-2][1]
        latest_rsi = swing_lows_rsi[-1][1]
        prior_rsi = swing_lows_rsi[-2][1]

        if latest_price < prior_price and latest_rsi > prior_rsi:
            if rsi_series.iloc[-1] < 50:
                results.append("BULLISH_DIVERGENCE")

    return results


def detect_volume_signals(close, volume, avg_period=None):
    """
    Analyze price/volume relationship for institutional activity.

    CORRECTED LOGIC (v1.5):
    - Distribution: DOWN days on HEAVY volume = institutions selling (bearish)
    - Accumulation: UP days on HEAVY volume = institutions buying (bullish)
    - Weak Rally: UP days on LOW volume = unsustainable retail buying (bearish)
    - Selling Pressure: Multiple down days with heavy volume (bearish)

    Returns: Dictionary of detected volume signals with details
    """
    if avg_period is None:
        avg_period = VOLUME_AVG_PERIOD

    if len(close) < avg_period + 3:
        return {}

    avg_vol = volume.rolling(avg_period).mean()
    recent_n = 5
    price_ch = close.diff()

    up_days = price_ch.iloc[-recent_n:] > 0
    down_days = price_ch.iloc[-recent_n:] < 0
    vol_recent = volume.iloc[-recent_n:]
    avg_recent = avg_vol.iloc[-recent_n:]

    signals = {}

    # BEARISH SIGNALS
    down_vol_heavy = (down_days & (vol_recent > avg_recent * 1.3)).sum()
    if down_vol_heavy >= 3:
        signals["DISTRIBUTION"] = f"Down days on heavy volume ({down_vol_heavy}/5 bars) - institutions selling"

    selling_pressure = (down_days & (vol_recent > avg_recent * 1.2)).sum()
    if selling_pressure >= 3 and "DISTRIBUTION" not in signals:
        signals["SELLING_PRESSURE"] = f"Sustained selling on volume ({selling_pressure}/5 bars)"

    up_vol_weak = (up_days & (vol_recent < avg_recent * 0.75)).sum()
    if up_vol_weak >= 3:
        signals["WEAK_RALLY"] = f"Up days on weak volume ({up_vol_weak}/5 bars) - unsustainable"

    # BULLISH SIGNALS
    up_vol_strong = (up_days & (vol_recent > avg_recent * 1.3)).sum()
    if up_vol_strong >= 3:
        signals["ACCUMULATION"] = f"Up days on heavy volume ({up_vol_strong}/5 bars) - institutions buying"

    down_vol_light = (down_days & (vol_recent < avg_recent * 0.75)).sum()
    if down_vol_light >= 3 and not signals:
        signals["BUYING_SUPPORT"] = f"Down days on light volume ({down_vol_light}/5 bars) - lack of selling"

    return signals


def detect_climax(close, volume, avg_period=None):
    """
    Detect climax top: large move on huge volume (exhaustion signal).
    Returns: (bool, detail_string)
    """
    if avg_period is None:
        avg_period = VOLUME_AVG_PERIOD

    if len(close) < avg_period + 2:
        return False, ""

    avg_vol = volume.rolling(avg_period).mean().iloc[-1]
    last_vol = volume.iloc[-1]
    last_move = abs((close.iloc[-1] - close.iloc[-2]) / close.iloc[-2])
    is_up_bar = close.iloc[-1] > close.iloc[-2]

    if (
        is_up_bar
        and last_vol > avg_vol * CLIMAX_VOLUME_MULT
        and last_move > CLIMAX_MOVE_PCT
    ):
        detail = f"Bar up {last_move*100:.1f}% on {last_vol/avg_vol:.1f}x avg volume"
        return True, detail

    return False, ""


def detect_failed_breakout(close, lookback=None):
    """
    Detect failed breakout: new high made, then price reversed below prior resistance.
    Returns: (bool, detail_string)
    """
    if lookback is None:
        lookback = BREAKOUT_LOOKBACK

    if len(close) < lookback + 3:
        return False, ""

    window = close.iloc[-(lookback+3):]
    prior_high = window.iloc[:-3].max()
    recent_high = window.iloc[-3:].max()
    current = close.iloc[-1]

    if recent_high > prior_high and current < prior_high:
        detail = f"Pushed to ${recent_high:.2f}, reversed below ${prior_high:.2f}"
        return True, detail

    return False, ""


def detect_ma_breakdown(close, ma_series, label="MA20"):
    """
    Detect moving average breakdown: 2 consecutive closes below MA after being above.
    Returns: (bool, detail_string)
    """
    if len(close) < 5 or len(ma_series) < 5:
        return False, ""

    was_above = (close.iloc[-5] > ma_series.iloc[-5])
    now_below_2 = (
        close.iloc[-1] < ma_series.iloc[-1]
        and close.iloc[-2] < ma_series.iloc[-2]
    )

    if was_above and now_below_2:
        return True, f"Two closes below {label} after being above"

    return False, ""


def score_reversal(reversal_signals):
    """
    Score reversal conviction based on number of confirming signals.
    Returns: (conviction_level, signal_count)
    """
    count = len(reversal_signals)
    if count >= 4:
        return "HIGH", count
    elif count >= 3:
        return "MODERATE", count
    elif count >= 2:
        return "WATCH", count
    return "NONE", count


print("✓ Reversal detection functions defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v1.5")

✓ Reversal detection functions defined
  Notebook Version: v2.5
  Cell Version: v1.5


## Text Cell 5: Position Sizing & Risk Management

### Position Sizing Logic
Contract allocation is determined by three factors:

1. **Signal Strength**: Strong signals use max risk (5%), weak signals use min risk (3%)
2. **Risk Budget**: Dollar amount = account size × risk percentage
3. **Concentration Limit**: Never exceed 20% of account in single position

### Safety Guardrails
- Minimum 1 contract if cost ≤ max risk budget
- Zero contracts if premium too expensive for risk parameters
- IV warning if percentile > 50th (consider waiting for cheaper entry)
- Position size display includes stop-loss and profit target levels

### Signal Strength Scoring
Combines multiple factors to classify setup quality:
- Base score from number of confirming signals
- +1 if clear trend (UPTREND/DOWNTREND)
- +1 if RSI extreme (< 30 or > 70)
- +1 if price significantly extended from MA20 (> 5%)

**Final Score**: 4+ = Strong | 2-3 = Moderate | < 2 = Weak

In [11]:
# Code Cell 5: Position Sizing Functions
# Notebook Version: v2.3
# Cell Version: v1.2

def position_sizing(account_size, option_ask, signal_strength, iv_percentile=None):
    """
    Calculate position size based on risk parameters and signal conviction.
    Returns: Dictionary with sizing details or None if invalid inputs
    """
    # ──────────────────────────────────────────────────────────────
    # INPUT VALIDATION
    # ──────────────────────────────────────────────────────────────
    if account_size <= 0:
        print("   ⚠ Error: Invalid account size")
        return None

    if option_ask <= 0 or np.isnan(option_ask) or np.isinf(option_ask):
        print("   ⚠ Error: Invalid option ask price")
        return None

    if signal_strength not in ["strong", "moderate", "weak"]:
        print(f"   ⚠ Warning: Unknown signal strength '{signal_strength}', using 'weak'")
        signal_strength = "weak"

    if iv_percentile is not None:
        if not (0 <= iv_percentile <= 100):
            print(f"   ⚠ Warning: IV percentile {iv_percentile} outside valid range")
            iv_percentile = None

    # ──────────────────────────────────────────────────────────────
    # POSITION SIZING CALCULATION
    # ──────────────────────────────────────────────────────────────
    tier_map = {
        "strong": RISK_PCT_MAX,
        "moderate": (RISK_PCT_MIN + RISK_PCT_MAX) / 2,
        "weak": RISK_PCT_MIN,
    }

    risk_pct = tier_map.get(signal_strength, RISK_PCT_MIN)
    risk_budget = round(account_size * risk_pct, 2)
    cost_per_contract = option_ask * 100

    max_allowed_cost = account_size * RISK_PCT_MAX
    if cost_per_contract > max_allowed_cost * 1.5:
        print(f"   ⚠ Warning: Option very expensive (${cost_per_contract:.0f}/contract)")

    max_contracts = int(risk_budget // cost_per_contract)

    if max_contracts < 1 and cost_per_contract <= account_size * RISK_PCT_MAX:
        max_contracts = 1

    max_by_concentration = int((account_size * 0.20) // cost_per_contract)
    contracts = max(min(max_contracts, max_by_concentration), 0)

    iv_note = ""
    if iv_percentile is not None and iv_percentile > IV_PCT_THRESHOLD:
        iv_note = f"IV elevated ({iv_percentile}th pctile) — consider 1 contract or wait"

    return {
        "risk_budget": risk_budget,
        "risk_pct": round(risk_pct * 100, 1),
        "cost_per_contract": cost_per_contract,
        "contracts": contracts,
        "total_cost": round(contracts * cost_per_contract, 2),
        "pct_of_account": round((contracts * cost_per_contract / account_size) * 100, 1) if contracts > 0 else 0,
        "stop_loss": round(option_ask * (1 - STOP_LOSS_PCT), 2),
        "profit_target": round(option_ask * (1 + PROFIT_TARGET_PCT), 2),
        "iv_note": iv_note,
    }


def signal_strength_score(signals, rsi, pct_from_ma20, trend):
    """
    Score signal strength based on multiple confirming factors.
    Returns: "strong", "moderate", "weak", or "none"
    """
    if not signals:
        return "none"

    score = len(signals)
    if trend in ("UPTREND", "DOWNTREND"):
        score += 1
    if rsi < 30 or rsi > 70:
        score += 1
    if abs(pct_from_ma20) > 5:
        score += 1

    if score >= 4:
        return "strong"
    elif score >= 2:
        return "moderate"
    else:
        return "weak"


def print_sizing(sizing, direction, option_ask):
    """
    Display formatted position sizing details with risk metrics.
    """
    print(f"\n ── POSITION SIZING ({direction}) ──")

    if not sizing or sizing["contracts"] == 0:
        print(f" Premium      : ${option_ask:.2f} (${option_ask*100:.0f}/contract)")
        print(f" Max budget   : ${round(ACCOUNT_SIZE * RISK_PCT_MAX, 2)}")
        print(f" ⚠ Too expensive for current risk budget")
        print(f" Suggestion   : Use debit spread to reduce cost")
        return

    print(f" Account      : ${ACCOUNT_SIZE:,.2f}")
    print(f" Risk Budget  : ${sizing['risk_budget']:,.2f} ({sizing['risk_pct']}% of account)")
    print(f" Premium      : ${option_ask:.2f} (${sizing['cost_per_contract']:.0f}/contract)")
    print(f" Contracts    : {sizing['contracts']}")
    print(f" Total Cost   : ${sizing['total_cost']:,.2f} ({sizing['pct_of_account']}% of account)")
    print(f" Stop Loss    : ${sizing['stop_loss']:.2f}/contract (exit at -{int(STOP_LOSS_PCT*100)}%)")
    print(f" Profit Target: ${sizing['profit_target']:.2f}/contract (exit at +{int(PROFIT_TARGET_PCT*100)}%)")

    if sizing["iv_note"]:
        print(f" ⚠ {sizing['iv_note']}")

    remaining = ACCOUNT_SIZE - sizing["total_cost"]
    print(f"\n ── ACCOUNT GUARDRAILS ──")
    print(f" Capital after trade: ${remaining:,.2f}")
    print(f" Max open positions : {MAX_POSITIONS}")

    if sizing["pct_of_account"] > 15:
        print(f" ⚠ {sizing['pct_of_account']}% of account in one trade — consider reducing size")


print("✓ Position sizing functions defined")
print("  Notebook Version: v2.3")
print("  Cell Version: v1.2")

✓ Position sizing functions defined
  Notebook Version: v2.3
  Cell Version: v1.2


## Text Cell 6: Options Chain Retrieval

### Helper Functions
These utilities interact with the yfinance API to fetch options data:

**get_best_expiry()**: Iterates through available expiration dates to find the first one meeting the minimum DTE requirement (45 days). Returns both the date string and calculated DTE.

**get_atm_options()**: Retrieves options chains for both calls and puts, then filters to strikes within 10% of current price. Sorts by absolute distance from price to prioritize true ATM strikes. Returns separate DataFrames for calls and puts.

### Error Handling
Both functions use try/except blocks to handle API errors gracefully. If yfinance fails to return data (market closed, delisted ticker, etc.), empty DataFrames are returned so the scanner can continue processing other tickers.

In [12]:
# Code Cell 6: Options Chain Helpers (v1.3 - Liquidity Filters)

def get_best_expiry(ticker_obj):
    """
    Find first available expiry date meeting minimum DTE requirement.
    Returns: (expiry_date_string, days_to_expiration) or (None, None)
    """
    try:
        today = datetime.today().date()

        # Validate ticker has options attribute
        if not hasattr(ticker_obj, 'options'):
            print("   ⚠ Ticker does not support options")
            return None, None

        # Check if options list is available
        if not ticker_obj.options or len(ticker_obj.options) == 0:
            print("   ⚠ No option expiration dates available")
            return None, None

        # Find first expiry meeting minimum DTE
        for exp in ticker_obj.options:
            try:
                exp_date = datetime.strptime(exp, "%Y-%m-%d").date()
                dte = (exp_date - today).days

                if dte >= MIN_DTE:
                    return exp, dte
            except ValueError:
                # Skip malformed dates
                continue

        # No expiry met minimum DTE requirement
        print(f"   ⚠ No expiry found with {MIN_DTE}+ DTE (nearest: {ticker_obj.options[0]})")
        return None, None

    except AttributeError as e:
        print(f"   ⚠ Error accessing options data: {e}")
        return None, None
    except Exception as e:
        print(f"   ⚠ Unexpected error in expiry lookup: {type(e).__name__}")
        return None, None


def get_atm_options(ticker_obj, expiry, price):
    """
    Retrieve near-ATM options with liquidity filtering and quality checks.
    Returns: (calls_dataframe, puts_dataframe) sorted by strike distance
    """
    try:
        # Fetch option chain
        chain = ticker_obj.option_chain(expiry)
        calls = chain.calls.copy()
        puts = chain.puts.copy()

        # Validate we got data
        if calls.empty and puts.empty:
            print("   ⚠ Options chain returned empty (market may be closed)")
            return pd.DataFrame(), pd.DataFrame()

        # ──────────────────────────────────────────────────────────
        # PROCESS CALLS
        # ──────────────────────────────────────────────────────────
        if not calls.empty:
            # Data quality filters
            calls = calls[
                (calls['bid'] > 0) &                    # Must have valid bid
                (calls['ask'] > calls['bid']) &         # Ask must be > bid
                (calls['ask'] < price * 0.5) &          # Sanity: call < 50% of stock
                (calls['lastPrice'] > 0)                # Must have traded
            ].copy()

            if not calls.empty:
                # Liquidity filters (NEW IN v1.3)
                calls = calls[
                    (calls['volume'].fillna(0) >= MIN_OPTION_VOLUME) |     # Volume check
                    (calls['openInterest'].fillna(0) >= MIN_OPTION_OI)     # OR open interest check
                ].copy()

                # Calculate distance from current price
                if not calls.empty:
                    calls['strike_dist'] = abs(calls['strike'] - price) / price
                    # Filter to ATM tolerance
                    calls = calls[calls['strike_dist'] < ATM_STRIKE_TOLERANCE].sort_values('strike_dist')

                    # Calculate bid/ask spread percentage
                    calls['spread_pct'] = ((calls['ask'] - calls['bid']) / calls['ask'] * 100)

        # ──────────────────────────────────────────────────────────
        # PROCESS PUTS
        # ──────────────────────────────────────────────────────────
        if not puts.empty:
            # Data quality filters
            puts = puts[
                (puts['bid'] > 0) &                     # Must have valid bid
                (puts['ask'] > puts['bid']) &           # Ask must be > bid
                (puts['ask'] < price * 0.5) &           # Sanity: put < 50% of stock
                (puts['lastPrice'] > 0)                 # Must have traded
            ].copy()

            if not puts.empty:
                # Liquidity filters (NEW IN v1.3)
                puts = puts[
                    (puts['volume'].fillna(0) >= MIN_OPTION_VOLUME) |      # Volume check
                    (puts['openInterest'].fillna(0) >= MIN_OPTION_OI)      # OR open interest check
                ].copy()

                # Calculate distance from current price
                if not puts.empty:
                    puts['strike_dist'] = abs(puts['strike'] - price) / price
                    # Filter to ATM tolerance
                    puts = puts[puts['strike_dist'] < ATM_STRIKE_TOLERANCE].sort_values('strike_dist')

                    # Calculate bid/ask spread percentage
                    puts['spread_pct'] = ((puts['ask'] - puts['bid']) / puts['ask'] * 100)

        # Final validation
        if calls.empty and puts.empty:
            print(f"   ⚠ No liquid ATM options (need {MIN_OPTION_VOLUME}+ volume OR {MIN_OPTION_OI}+ OI)")

        return calls, puts

    except AttributeError as e:
        print(f"   ⚠ Error: Invalid option chain format ({e})")
        return pd.DataFrame(), pd.DataFrame()
    except KeyError as e:
        print(f"   ⚠ Error: Missing expected column in options data ({e})")
        return pd.DataFrame(), pd.DataFrame()
    except Exception as e:
        print(f"   ⚠ Unexpected error retrieving options: {type(e).__name__}")
        return pd.DataFrame(), pd.DataFrame()


print("✓ Options chain functions defined (v1.3 - Liquidity filters)")

✓ Options chain functions defined (v1.3 - Liquidity filters)


## Text Cell 6A: Trade Journal Export

### Purpose
The trade journal automatically logs every scan to a CSV file, creating a permanent record of all signals, market conditions, and position sizing decisions.

### What Gets Logged
- **Timestamp**: Exact date/time of scan
- **Market Data**: Price, RSI, trend, distance from MAs
- **Signals**: Direction (CALL/PUT), conviction level, signal descriptions
- **Reversals**: Conviction score, reversal pattern details
- **Options Data**: Strike, ask price, IV, Delta, Theta
- **Position Sizing**: Contracts, total cost, risk %, stop/target

### Usage
- **Enable/Disable**: Set `ENABLE_JOURNAL = True/False` in Code Cell 2
- **File Location**: `JOURNAL_PATH = "trade_journal.csv"` (same folder as notebook)
- **View Data**: Download the CSV and open in Excel/Google Sheets
- **Analysis**: Track which setups work, win rate by conviction level, average returns

### Benefits
- Historical record of all scanner outputs
- Track performance over time
- Identify which signal combinations work best
- Backtest strategy refinements
- Tax documentation for actual trades taken

In [13]:
# Code Cell 6A: Trade Journal Functions
# Notebook Version: v2.5
# Cell Version: v2.5

import os
from datetime import datetime
from pytz import timezone as pytz_timezone

def log_to_journal(scan_data):
    """
    Append scanner results to CSV trade journal.
    """
    if not ENABLE_JOURNAL:
        return

    try:
        file_exists = os.path.isfile(JOURNAL_PATH)
        df = pd.DataFrame([scan_data])

        df.to_csv(
            JOURNAL_PATH,
            mode='a',
            header=not file_exists,
            index=False
        )

    except Exception as e:
        print(f"   ⚠ Journal logging failed: {e}")


def get_local_timestamp():
    """
    Get current timestamp in configured timezone.
    """
    try:
        from datetime import datetime, timezone as dt_timezone
        utc_now = datetime.now(dt_timezone.utc)
        tz = pytz_timezone(TIMEZONE)
        local_time = utc_now.astimezone(tz)
        return local_time.strftime('%Y-%m-%d %H:%M:%S %Z')
    except Exception:
        return datetime.now().strftime('%Y-%m-%d %H:%M:%S')


def create_journal_entry(symbol, price, rsi, trend, market_regime, p_ma20, p_ma50,
                        reversal_signals, signals, strength,
                        expiry, dte, option_data, sizing):
    """
    Format scanner results into journal entry dictionary.
    """
    timestamp = get_local_timestamp()

    signal_list = [s[1] for s in signals] if signals else []
    reversal_list = [r[1] for r in reversal_signals] if reversal_signals else []

    directions = list(dict.fromkeys([s[0] for s in signals])) if signals else []
    primary_direction = directions[0] if directions else "NO SIGNAL"

    rev_conviction, rev_count = score_reversal(reversal_signals if reversal_signals else [])

    entry = {
        'timestamp': timestamp,
        'symbol': symbol,
        'price': price,
        'rsi': rsi,
        'trend': trend,
        'market_regime': market_regime,
        'pct_from_ma20': p_ma20,
        'pct_from_ma50': p_ma50,

        'direction': primary_direction,
        'conviction': strength,
        'signal_count': len(signals),
        'signals': ' | '.join(signal_list),

        'reversal_conviction': rev_conviction,
        'reversal_count': rev_count,
        'reversals': ' | '.join(reversal_list),

        'expiry': expiry if expiry else 'N/A',
        'dte': dte if dte else 0,
        'strike': option_data.get('strike', 'N/A') if option_data else 'N/A',
        'option_ask': option_data.get('ask', 0) if option_data else 0,
        'option_iv': option_data.get('iv', 0) if option_data else 0,
        'option_delta': option_data.get('delta', 0) if option_data else 0,
        'option_theta': option_data.get('theta', 0) if option_data else 0,

        'contracts': sizing.get('contracts', 0) if sizing else 0,
        'total_cost': sizing.get('total_cost', 0) if sizing else 0,
        'risk_pct': sizing.get('risk_pct', 0) if sizing else 0,
        'stop_loss': sizing.get('stop_loss', 0) if sizing else 0,
        'profit_target': sizing.get('profit_target', 0) if sizing else 0,
    }

    return entry


print("✓ Trade journal functions defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Trade journal functions defined
  Notebook Version: v2.5
  Cell Version: v2.5


In [14]:
# Code Cell 6B: Visualization Functions
# Notebook Version: v2.3
# Cell Version: v1.7

def plot_analysis(symbol, hist, close, rsi_series, ma_s, ma_l,
                 reversal_signals, signals, price):
    """
    Create price chart with technical indicators and signal markers.

    Args:
        symbol: Ticker symbol
        hist: Historical price data
        close: Close price series
        rsi_series: RSI values
        ma_s: Short MA series
        ma_l: Long MA series
        reversal_signals: List of detected reversal patterns
        signals: List of directional signals
        price: Current price
    """
    if not ENABLE_CHARTS:
        return

    try:
        days_to_plot = min(CHART_PERIOD, len(hist))
        plot_data = hist.iloc[-days_to_plot:].copy()

        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(14, 8),
            gridspec_kw={'height_ratios': [3, 1]}
        )

        # ──────────────────────────────────────────────────────────
        # UPPER PANEL: PRICE & MOVING AVERAGES
        # ──────────────────────────────────────────────────────────
        ax1.plot(plot_data.index, plot_data['Close'],
                 label='Price', color='black', linewidth=1.5)
        ax1.plot(plot_data.index, ma_s.iloc[-days_to_plot:],
                 label=f'MA{MA_SHORT}', color='blue', linewidth=1, alpha=0.7)
        ax1.plot(plot_data.index, ma_l.iloc[-days_to_plot:],
                 label=f'MA{MA_LONG}', color='red', linewidth=1, alpha=0.7)

        ax1.axhline(y=price, color='gray', linestyle='--',
                    linewidth=0.8, alpha=0.5, label=f'Current: ${price}')

        if signals:
            signal_date = plot_data.index[-1]
            for direction, reason in signals:
                if direction == "BUY CALL":
                    ax1.scatter(signal_date, price, color='green',
                                marker='^', s=200, zorder=5,
                                label='CALL Signal', edgecolors='black')
                else:
                    ax1.scatter(signal_date, price, color='red',
                                marker='v', s=200, zorder=5,
                                label='PUT Signal', edgecolors='black')

        if reversal_signals:
            signal_date = plot_data.index[-1]
            bearish_count = sum(1 for r in reversal_signals if r[0] == "BEARISH")
            bullish_count = sum(1 for r in reversal_signals if r[0] == "BULLISH")

            if bearish_count > 0:
                ax1.scatter(signal_date, price * 1.02, color='orange',
                            marker='v', s=100, alpha=0.7,
                            label=f'{bearish_count} Bearish Reversals')
            if bullish_count > 0:
                ax1.scatter(signal_date, price * 0.98, color='cyan',
                            marker='^', s=100, alpha=0.7,
                            label=f'{bullish_count} Bullish Reversals')

        ax1.set_title(f'{symbol} - Price Analysis', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Price ($)', fontsize=11)
        ax1.legend(loc='best', fontsize=9)
        ax1.grid(True, alpha=0.3)

        # ──────────────────────────────────────────────────────────
        # LOWER PANEL: RSI
        # ──────────────────────────────────────────────────────────
        ax2.plot(plot_data.index, rsi_series.iloc[-days_to_plot:],
                 color='purple', linewidth=1.5, label='RSI(14)')

        ax2.axhline(y=RSI_OVERBOUGHT, color='red', linestyle='--',
                    linewidth=0.8, alpha=0.5, label=f'Overbought ({RSI_OVERBOUGHT})')
        ax2.axhline(y=RSI_OVERSOLD, color='green', linestyle='--',
                    linewidth=0.8, alpha=0.5, label=f'Oversold ({RSI_OVERSOLD})')
        ax2.axhline(y=50, color='gray', linestyle='-',
                    linewidth=0.5, alpha=0.3)

        ax2.fill_between(plot_data.index, RSI_OVERBOUGHT, 100, alpha=0.1, color='red')
        ax2.fill_between(plot_data.index, 0, RSI_OVERSOLD, alpha=0.1, color='green')

        ax2.set_ylabel('RSI', fontsize=11)
        ax2.set_xlabel('Date', fontsize=11)
        ax2.set_ylim([0, 100])
        ax2.legend(loc='best', fontsize=9)
        ax2.grid(True, alpha=0.3)

        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
        ax2.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"   ⚠ Chart generation failed: {e}")


print("✓ Visualization functions defined")
print("  Notebook Version: v2.3")
print("  Cell Version: v1.7")

✓ Visualization functions defined
  Notebook Version: v2.3
  Cell Version: v1.7


## Text Cell 7: Main Scanner Function

### Workflow Overview
The `evaluate_ticker()` function orchestrates the complete analysis pipeline for each symbol:

**1. Data Retrieval**: Fetch 1-year price history via yfinance, validate sufficient data exists

**2. Technical Analysis**: Calculate RSI, moving averages, historical volatility, trend classification

**3. Market Regime Check**:
- Analyze benchmark market regime using SPY
- Classify broader market as BULLISH, MIXED, or BEARISH
- Use regime context to improve trade selection

**4. Reversal Detection**: Run all pattern detection functions, collect signals, score conviction

**5. Signal Generation**: Evaluate both call and put setups based on:
- Standard technical conditions (RSI extremes, trend alignment, extension levels)
- Reversal-driven opportunities (uptrend showing cracks, bullish divergence in downtrend)
- Bearish-regime downside participation when broader conditions support PUTs

**6. Regime Filtering**:
- Filter most bullish CALL setups when the broader market is BEARISH
- Filter `Clean uptrend` CALL setups when the broader market is MIXED
- Preserve only the stronger selective CALL exceptions in non-bullish conditions

**7. Signal Strength Boosting**: If reversal signals align with directional bias, upgrade conviction level

**8. Options Chain Analysis**: Retrieve best expiry, find ATM strikes, display bid/ask/IV/OI

**9. Position Sizing**: Calculate contracts, total cost, stop/target levels, display formatted output

### Output Sections
- Market Snapshot (price, RSI, trend, moving averages, volatility)
- Market Regime (benchmark regime, MA structure)
- Reversal Watch (all detected patterns with conviction score)
- Signals (directional setups with entry rationale)
- Options Chain Preview (strike, bid/ask, IV, open interest)
- Position Sizing (contracts, cost, risk metrics, account impact)

In [15]:
# Code Cell 7: Main Scanner Function
# Notebook Version: v2.5
# Cell Version: v2.5

def evaluate_ticker(symbol):
    """
    Complete analysis pipeline for single ticker: fetch data, detect patterns,
    generate signals, apply regime filtering, size positions, display results,
    and log to journal.
    """
    print(f"\n{'='*62}")
    print(f" SCANNING: {symbol}")
    print(f"{'='*62}")

    # ──────────────────────────────────────────────────────────────
    # DATA RETRIEVAL & VALIDATION
    # ──────────────────────────────────────────────────────────────
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period="1y")
    except Exception as e:
        print(f" ⚠ Error fetching data for {symbol}: {type(e).__name__}")
        return

    if hist.empty or len(hist) < MA_LONG + 5:
        print(" ⚠ Insufficient price history (need 1 year of data)")
        return

    required_cols = ['Close', 'Volume']
    if not all(col in hist.columns for col in required_cols):
        print(" ⚠ Missing required price data columns")
        return

    close = hist['Close']
    volume = hist['Volume']

    if close.isna().sum() > len(close) * 0.1:
        print(f" ⚠ Warning: {close.isna().sum()} missing price values")

    price = round(close.iloc[-1], 2)

    if price <= 0 or np.isnan(price):
        print(" ⚠ Invalid current price")
        return

    # ──────────────────────────────────────────────────────────────
    # TECHNICAL INDICATORS
    # ──────────────────────────────────────────────────────────────
    rsi_s = compute_rsi(close)
    rsi = round(rsi_s.iloc[-1], 1)

    if np.isnan(rsi):
        print(" ⚠ Unable to calculate RSI (insufficient data)")
        return

    ma_s = close.rolling(MA_SHORT).mean()
    ma_l = close.rolling(MA_LONG).mean()
    ma_s_current = ma_s.iloc[-1]
    ma_l_current = ma_l.iloc[-1]

    trend = get_trend(price, ma_s_current, ma_l_current)

    hv_s = compute_historical_volatility(close)
    hv = round(hv_s.iloc[-1], 1) if not np.isnan(hv_s.iloc[-1]) else 0
    iv_pct = compute_iv_percentile(hv_s.dropna(), hv)

    p_ma20 = round((price - ma_s_current) / ma_s_current * 100, 2)
    p_ma50 = round((price - ma_l_current) / ma_l_current * 100, 2)

    expiry, dte = get_best_expiry(ticker)

    # ──────────────────────────────────────────────────────────────
    # WEEKLY TIMEFRAME ANALYSIS
    # ──────────────────────────────────────────────────────────────
    weekly_data = None
    if ENABLE_MTF:
        weekly_data = analyze_weekly_timeframe(ticker)

    # ──────────────────────────────────────────────────────────────
    # MARKET REGIME ANALYSIS
    # ──────────────────────────────────────────────────────────────
    market_regime_data = None
    market_regime = "UNKNOWN"

    if ENABLE_REGIME_FILTER:
        market_regime_data = analyze_market_regime(REGIME_SYMBOL)
        if market_regime_data:
            market_regime = market_regime_data['regime']

    # ──────────────────────────────────────────────────────────────
    # MARKET SNAPSHOT
    # ──────────────────────────────────────────────────────────────
    print(f"\n ── DAILY TIMEFRAME ──")
    print(f" Price       : ${price}")
    print(f" RSI (14)    : {rsi}")
    print(f" Trend       : {trend}")
    print(f" MA20        : ${round(ma_s_current, 2)} ({p_ma20:+.1f}%)")
    print(f" MA50        : ${round(ma_l_current, 2)} ({p_ma50:+.1f}%)")
    print(f" 30d HV      : {hv}%")
    print(f" IV Pctile   : {iv_pct}th" if iv_pct is not None else " IV Pctile   : N/A")
    print(f" Best Expiry : {expiry} ({dte} DTE)" if expiry else " Best Expiry : None found")

    if weekly_data:
        print(f"\n ── WEEKLY TIMEFRAME ──")
        print(f" Weekly RSI  : {weekly_data['rsi']}")
        print(f" Weekly Trend: {weekly_data['trend']}")
        print(f" MA{WEEKLY_MA_SHORT}W       : ${weekly_data['ma_short']} ({weekly_data['pct_from_ma_short']:+.1f}%)")
        print(f" MA{WEEKLY_MA_LONG}W       : ${weekly_data['ma_long']} ({weekly_data['pct_from_ma_long']:+.1f}%)")

    if market_regime_data:
        print(f"\n ── MARKET REGIME ({REGIME_SYMBOL}) ──")
        print(f" Regime      : {market_regime}")
        print(f" Price       : ${market_regime_data['price']}")
        print(f" MA{REGIME_MA_SHORT}        : ${market_regime_data['ma_short']} ({market_regime_data['pct_from_ma_short']:+.1f}%)")
        print(f" MA{REGIME_MA_LONG}       : ${market_regime_data['ma_long']} ({market_regime_data['pct_from_ma_long']:+.1f}%)")

    # ──────────────────────────────────────────────────────────────
    # REVERSAL DETECTION
    # ──────────────────────────────────────────────────────────────
    divergences = detect_rsi_divergence(close, rsi_s)
    vol_signals = detect_volume_signals(close, volume)
    climax, c_det = detect_climax(close, volume)
    failed_bo, f_det = detect_failed_breakout(close)
    ma_break, m_det = detect_ma_breakdown(close, ma_s, "MA20")

    reversal_signals = []

    if "BEARISH_DIVERGENCE" in divergences:
        reversal_signals.append(("BEARISH", "RSI bearish divergence — price up, momentum down"))
    if "BULLISH_DIVERGENCE" in divergences:
        reversal_signals.append(("BULLISH", "RSI bullish divergence — price down, momentum up"))

    if "DISTRIBUTION" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['DISTRIBUTION']))
    if "SELLING_PRESSURE" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['SELLING_PRESSURE']))
    if "WEAK_RALLY" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['WEAK_RALLY']))
    if "ACCUMULATION" in vol_signals:
        reversal_signals.append(("BULLISH", vol_signals['ACCUMULATION']))
    if "BUYING_SUPPORT" in vol_signals:
        reversal_signals.append(("BULLISH", vol_signals['BUYING_SUPPORT']))

    if climax:
        reversal_signals.append(("BEARISH", f"Climax top — {c_det}"))
    if failed_bo:
        reversal_signals.append(("BEARISH", f"Failed breakout — {f_det}"))
    if ma_break:
        reversal_signals.append(("BEARISH", f"MA breakdown — {m_det}"))

    bearish_rev = [r for r in reversal_signals if r[0] == "BEARISH"]
    bullish_rev = [r for r in reversal_signals if r[0] == "BULLISH"]
    rev_conviction, rev_count = score_reversal(bearish_rev)

    print(f"\n ── REVERSAL WATCH ──")
    if reversal_signals:
        if bearish_rev:
            print(f" Bearish signals [{rev_conviction}, {rev_count} confirming]:")
            for _, detail in bearish_rev:
                print(f"   ⚠ {detail}")
        if bullish_rev:
            print(" Bullish signals:")
            for _, detail in bullish_rev:
                print(f"   ✓ {detail}")
    else:
        print(" No reversal signals detected")

    # ──────────────────────────────────────────────────────────────
    # DIRECTIONAL SIGNALS
    # ──────────────────────────────────────────────────────────────
    signals = []

    # CALL signals
    if rsi < RSI_OVERSOLD and trend in ("UPTREND", "MIXED") and p_ma50 > -8:
        signals.append(("BUY CALL", "RSI oversold + holding MA50 support"))

    if trend == "UPTREND" and 0 < p_ma20 < 5 and rsi < 60 and not bearish_rev:
        signals.append(("BUY CALL", "Clean uptrend, not extended, no reversal signals"))

    if bullish_rev and rsi < 50 and trend != "DOWNTREND":
        signals.append(("BUY CALL", "Bullish reversal signals present"))

    # PUT signals
    if trend == "UPTREND" and rev_conviction == "HIGH":
        signals.append(("BUY PUT", f"Uptrend reversal - HIGH conviction [{rev_count} signals]"))

    if rsi > RSI_OVERBOUGHT and trend == "DOWNTREND" and p_ma50 < -5:
        signals.append(("BUY PUT", "RSI overbought in confirmed downtrend"))

    if ENABLE_REGIME_FILTER and market_regime == "BEARISH":
        if trend == "DOWNTREND" and bearish_rev and rev_conviction in ("MODERATE", "HIGH"):
            signals.append(("BUY PUT", f"Bearish regime + bearish reversals [{rev_count} signals]"))

    # De-dupe
    deduped_signals = []
    seen = set()
    for sig in signals:
        if sig not in seen:
            deduped_signals.append(sig)
            seen.add(sig)
    signals = deduped_signals

    # ──────────────────────────────────────────────────────────────
    # MULTI-TIMEFRAME FILTERING
    # ──────────────────────────────────────────────────────────────
    if signals and weekly_data and ENABLE_MTF:
        weekly_trend = weekly_data['trend']
        filtered_signals = []

        for direction, reason in signals:
            is_aligned, confidence_boost, mtf_desc = check_timeframe_alignment(
                trend, weekly_trend, direction
            )

            if REQUIRE_MTF_ALIGNMENT and not is_aligned:
                print(f"\n ⚠ FILTERED (MTF): {direction} - {mtf_desc}")
                continue

            filtered_signals.append((direction, reason))

            if confidence_boost != 0:
                print(f"\n {mtf_desc}")

        signals = filtered_signals

    # ──────────────────────────────────────────────────────────────
    # MARKET REGIME FILTERING
    # ──────────────────────────────────────────────────────────────
    signals, regime_notes = filter_signals_for_regime(
        signals, market_regime, trend, rsi, bullish_rev
    )

    if regime_notes:
        print(f"\n ── REGIME FILTER NOTES ──")
        for note in regime_notes:
            print(f" ⚠ {note}")

    # Calculate signal strength
    strength = signal_strength_score(signals, rsi, p_ma20, trend)

    put_signals = [s for s in signals if s[0] == "BUY PUT"]
    if put_signals and rev_conviction in ("MODERATE", "HIGH"):
        if strength == "moderate":
            strength = "strong"
        elif strength == "weak":
            strength = "moderate"

    if signals and weekly_data and ENABLE_MTF:
        for direction, _ in signals:
            _, confidence_boost, _ = check_timeframe_alignment(
                trend, weekly_data['trend'], direction
            )
            if confidence_boost > 0:
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break

    if signals and ENABLE_REGIME_FILTER:
        for direction, reason in signals:
            if market_regime == "BULLISH" and direction == "BUY CALL":
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break
            if market_regime == "BEARISH" and direction == "BUY PUT":
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break

    print(f"\n ── SIGNALS [Conviction: {strength.upper()}] ──")
    if not signals:
        print(" No high-probability setup today")

        plot_analysis(symbol, hist, close, rsi_s, ma_s, ma_l,
                      reversal_signals, signals, price)

        if ENABLE_JOURNAL:
            journal_entry = create_journal_entry(
                symbol, price, rsi, trend, market_regime, p_ma20, p_ma50,
                reversal_signals, [], "none",
                expiry, dte, None, None
            )
            log_to_journal(journal_entry)

        return

    for direction, reason in signals:
        print(f" + {direction} — {reason}")

    if expiry:
        print(f" Target: {expiry} ({dte} DTE), near-ATM strike")

    # ──────────────────────────────────────────────────────────────
    # VISUALIZATION
    # ──────────────────────────────────────────────────────────────
    plot_analysis(symbol, hist, close, rsi_s, ma_s, ma_l,
                  reversal_signals, signals, price)

    if not expiry:
        print("\n No valid expiry found — cannot size position")
        return

    # ──────────────────────────────────────────────────────────────
    # OPTIONS CHAIN & POSITION SIZING
    # ──────────────────────────────────────────────────────────────
    atm_calls, atm_puts = get_atm_options(ticker, expiry, price)
    time_to_expiry = dte / 365.0 if dte else 0.123

    directions_in_order = []
    for d, _ in signals:
        if d not in directions_in_order:
            directions_in_order.append(d)

    for direction in directions_in_order:
        chain_df = atm_calls if direction == "BUY CALL" else atm_puts
        label = "CALL" if direction == "BUY CALL" else "PUT"
        option_type = "call" if direction == "BUY CALL" else "put"

        if chain_df.empty:
            print(f"\n ⚠ No liquid {label} options available")
            print(f"    (Need {MIN_OPTION_VOLUME}+ volume OR {MIN_OPTION_OI}+ open interest)")
            continue

        best = chain_df.iloc[0]
        option_ask = best.get('ask', 0)
        option_bid = best.get('bid', 0)
        option_iv = round(best.get('impliedVolatility', 0) * 100, 1)
        option_oi = int(best.get('openInterest', 0))
        option_volume = int(best.get('volume', 0)) if pd.notna(best.get('volume')) else 0
        spread_pct = best.get('spread_pct', 0)

        greeks = get_greeks_for_option(best, price, time_to_expiry, option_type)
        delta = greeks['delta']
        gamma = greeks['gamma']
        theta = greeks['theta']
        vega = greeks['vega']

        print(f"\n ── {label} CHAIN PREVIEW ({expiry}) ──")
        print(f" Strike     : ${best['strike']}")
        print(f" Bid/Ask    : ${option_bid:.2f} / ${option_ask:.2f} (spread: {spread_pct:.1f}%)")
        print(f" IV         : {option_iv}%")
        print(f" Volume     : {option_volume}")
        print(f" Open Int   : {option_oi}")

        print(f"\n ── GREEKS ──")
        if not np.isnan(delta):
            print(f" Delta      : {delta:.3f} (${abs(delta)*100:.0f} exposure per $1 stock move)")
        else:
            print(" Delta      : N/A")

        if not np.isnan(gamma):
            print(f" Gamma      : {gamma:.4f}")
        else:
            print(" Gamma      : N/A")

        if not np.isnan(theta):
            theta_per_day = theta * 100
            print(f" Theta      : {theta:.3f} (${theta_per_day:.2f}/day time decay)")
            if abs(theta_per_day) > option_ask * 100 * 0.05:
                print("            → ⚠ High time decay, need quick move")
        else:
            print(" Theta      : N/A")

        if not np.isnan(vega):
            vega_dollars = vega * 100
            print(f" Vega       : {vega:.3f} (${vega_dollars:.2f} per 1% IV change)")
            if vega_dollars > option_ask * 100 * 0.10:
                print("            → High IV sensitivity, watch for IV crush")
        else:
            print(" Vega       : N/A")

        liquidity_issues = []

        if option_volume < MIN_OPTION_VOLUME and option_oi < MIN_OPTION_OI:
            liquidity_issues.append(f"Low liquidity (vol: {option_volume}, OI: {option_oi})")

        if spread_pct > MAX_SPREAD_PCT:
            liquidity_issues.append(f"Wide spread ({spread_pct:.1f}% > {MAX_SPREAD_PCT}%)")

        if option_volume == 0:
            liquidity_issues.append("No volume today (stale pricing)")

        if liquidity_issues:
            print(f"\n ⚠ LIQUIDITY WARNINGS:")
            for issue in liquidity_issues:
                print(f"   • {issue}")
            print("   → Consider next expiry or different strike")

        if option_ask <= 0:
            print(" ⚠ No valid ask price — market may be closed")
            continue

        sizing = position_sizing(ACCOUNT_SIZE, option_ask, strength, iv_pct)

        if sizing:
            print_sizing(sizing, direction, option_ask)

            if not np.isnan(theta) and not np.isnan(delta) and sizing["contracts"] > 0:
                total_theta = theta * 100 * sizing["contracts"]
                total_delta_exposure = abs(delta) * price * sizing["contracts"]

                print(f"\n ── POSITION GREEKS SUMMARY ──")
                print(f" Total Delta Exposure: ${total_delta_exposure:,.0f}")
                print(f" Daily Theta Cost    : ${abs(total_theta):.2f}/day")

                if abs(delta) > 0.01:
                    breakeven_move = abs(total_theta / (abs(delta) * sizing['contracts']))
                    print(f" Break-even Move     : ${breakeven_move:.2f}/day in stock")

                days_to_expiry = dte if dte else 45
                total_decay = abs(total_theta) * days_to_expiry
                if total_decay > sizing["total_cost"] * 0.5:
                    print(f" ⚠ Time decay will erode {total_decay/sizing['total_cost']*100:.0f}% of premium by expiry")

            if ENABLE_JOURNAL:
                option_data = {
                    'strike': best['strike'],
                    'ask': option_ask,
                    'iv': option_iv,
                    'delta': delta if not np.isnan(delta) else 0,
                    'theta': theta if not np.isnan(theta) else 0
                }

                journal_entry = create_journal_entry(
                    symbol, price, rsi, trend, market_regime, p_ma20, p_ma50,
                    reversal_signals, signals, strength,
                    expiry, dte, option_data, sizing
                )
                log_to_journal(journal_entry)
                print(f"\n ✓ Logged to journal: {JOURNAL_PATH}")

        else:
            print(" ⚠ Unable to calculate position size (invalid inputs)")


print("✓ Main scanner function defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Main scanner function defined
  Notebook Version: v2.5
  Cell Version: v2.5


In [16]:
# Code Cell 7: Main Scanner Function
# Notebook Version: v2.5
# Cell Version: v2.5

def evaluate_ticker(symbol):
    """
    Complete analysis pipeline for single ticker: fetch data, detect patterns,
    generate signals, apply regime filtering, size positions, display results,
    and log to journal.
    """
    print(f"\n{'='*62}")
    print(f" SCANNING: {symbol}")
    print(f"{'='*62}")

    # ──────────────────────────────────────────────────────────────
    # DATA RETRIEVAL & VALIDATION
    # ──────────────────────────────────────────────────────────────
    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(period="1y")
    except Exception as e:
        print(f" ⚠ Error fetching data for {symbol}: {type(e).__name__}")
        return

    if hist.empty or len(hist) < MA_LONG + 5:
        print(" ⚠ Insufficient price history (need 1 year of data)")
        return

    required_cols = ['Close', 'Volume']
    if not all(col in hist.columns for col in required_cols):
        print(" ⚠ Missing required price data columns")
        return

    close = hist['Close']
    volume = hist['Volume']

    if close.isna().sum() > len(close) * 0.1:
        print(f" ⚠ Warning: {close.isna().sum()} missing price values")

    price = round(close.iloc[-1], 2)

    if price <= 0 or np.isnan(price):
        print(" ⚠ Invalid current price")
        return

    # ──────────────────────────────────────────────────────────────
    # TECHNICAL INDICATORS
    # ──────────────────────────────────────────────────────────────
    rsi_s = compute_rsi(close)
    rsi = round(rsi_s.iloc[-1], 1)

    if np.isnan(rsi):
        print(" ⚠ Unable to calculate RSI (insufficient data)")
        return

    ma_s = close.rolling(MA_SHORT).mean()
    ma_l = close.rolling(MA_LONG).mean()
    ma_s_current = ma_s.iloc[-1]
    ma_l_current = ma_l.iloc[-1]

    trend = get_trend(price, ma_s_current, ma_l_current)

    hv_s = compute_historical_volatility(close)
    hv = round(hv_s.iloc[-1], 1) if not np.isnan(hv_s.iloc[-1]) else 0
    iv_pct = compute_iv_percentile(hv_s.dropna(), hv)

    p_ma20 = round((price - ma_s_current) / ma_s_current * 100, 2)
    p_ma50 = round((price - ma_l_current) / ma_l_current * 100, 2)

    expiry, dte = get_best_expiry(ticker)

    # ──────────────────────────────────────────────────────────────
    # WEEKLY TIMEFRAME ANALYSIS
    # ──────────────────────────────────────────────────────────────
    weekly_data = None
    if ENABLE_MTF:
        weekly_data = analyze_weekly_timeframe(ticker)

    # ──────────────────────────────────────────────────────────────
    # MARKET REGIME ANALYSIS
    # ──────────────────────────────────────────────────────────────
    market_regime_data = None
    market_regime = "UNKNOWN"

    if ENABLE_REGIME_FILTER:
        market_regime_data = analyze_market_regime(REGIME_SYMBOL)
        if market_regime_data:
            market_regime = market_regime_data['regime']

    # ──────────────────────────────────────────────────────────────
    # MARKET SNAPSHOT
    # ──────────────────────────────────────────────────────────────
    print(f"\n ── DAILY TIMEFRAME ──")
    print(f" Price       : ${price}")
    print(f" RSI (14)    : {rsi}")
    print(f" Trend       : {trend}")
    print(f" MA20        : ${round(ma_s_current, 2)} ({p_ma20:+.1f}%)")
    print(f" MA50        : ${round(ma_l_current, 2)} ({p_ma50:+.1f}%)")
    print(f" 30d HV      : {hv}%")
    print(f" IV Pctile   : {iv_pct}th" if iv_pct is not None else " IV Pctile   : N/A")
    print(f" Best Expiry : {expiry} ({dte} DTE)" if expiry else " Best Expiry : None found")

    if weekly_data:
        print(f"\n ── WEEKLY TIMEFRAME ──")
        print(f" Weekly RSI  : {weekly_data['rsi']}")
        print(f" Weekly Trend: {weekly_data['trend']}")
        print(f" MA{WEEKLY_MA_SHORT}W       : ${weekly_data['ma_short']} ({weekly_data['pct_from_ma_short']:+.1f}%)")
        print(f" MA{WEEKLY_MA_LONG}W       : ${weekly_data['ma_long']} ({weekly_data['pct_from_ma_long']:+.1f}%)")

    if market_regime_data:
        print(f"\n ── MARKET REGIME ({REGIME_SYMBOL}) ──")
        print(f" Regime      : {market_regime}")
        print(f" Price       : ${market_regime_data['price']}")
        print(f" MA{REGIME_MA_SHORT}        : ${market_regime_data['ma_short']} ({market_regime_data['pct_from_ma_short']:+.1f}%)")
        print(f" MA{REGIME_MA_LONG}       : ${market_regime_data['ma_long']} ({market_regime_data['pct_from_ma_long']:+.1f}%)")

    # ──────────────────────────────────────────────────────────────
    # REVERSAL DETECTION
    # ──────────────────────────────────────────────────────────────
    divergences = detect_rsi_divergence(close, rsi_s)
    vol_signals = detect_volume_signals(close, volume)
    climax, c_det = detect_climax(close, volume)
    failed_bo, f_det = detect_failed_breakout(close)
    ma_break, m_det = detect_ma_breakdown(close, ma_s, "MA20")

    reversal_signals = []

    if "BEARISH_DIVERGENCE" in divergences:
        reversal_signals.append(("BEARISH", "RSI bearish divergence — price up, momentum down"))
    if "BULLISH_DIVERGENCE" in divergences:
        reversal_signals.append(("BULLISH", "RSI bullish divergence — price down, momentum up"))

    if "DISTRIBUTION" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['DISTRIBUTION']))
    if "SELLING_PRESSURE" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['SELLING_PRESSURE']))
    if "WEAK_RALLY" in vol_signals:
        reversal_signals.append(("BEARISH", vol_signals['WEAK_RALLY']))
    if "ACCUMULATION" in vol_signals:
        reversal_signals.append(("BULLISH", vol_signals['ACCUMULATION']))
    if "BUYING_SUPPORT" in vol_signals:
        reversal_signals.append(("BULLISH", vol_signals['BUYING_SUPPORT']))

    if climax:
        reversal_signals.append(("BEARISH", f"Climax top — {c_det}"))
    if failed_bo:
        reversal_signals.append(("BEARISH", f"Failed breakout — {f_det}"))
    if ma_break:
        reversal_signals.append(("BEARISH", f"MA breakdown — {m_det}"))

    bearish_rev = [r for r in reversal_signals if r[0] == "BEARISH"]
    bullish_rev = [r for r in reversal_signals if r[0] == "BULLISH"]
    rev_conviction, rev_count = score_reversal(bearish_rev)

    print(f"\n ── REVERSAL WATCH ──")
    if reversal_signals:
        if bearish_rev:
            print(f" Bearish signals [{rev_conviction}, {rev_count} confirming]:")
            for _, detail in bearish_rev:
                print(f"   ⚠ {detail}")
        if bullish_rev:
            print(" Bullish signals:")
            for _, detail in bullish_rev:
                print(f"   ✓ {detail}")
    else:
        print(" No reversal signals detected")

    # ──────────────────────────────────────────────────────────────
    # DIRECTIONAL SIGNALS
    # ──────────────────────────────────────────────────────────────
    signals = []

    # CALL signals
    if rsi < RSI_OVERSOLD and trend in ("UPTREND", "MIXED") and p_ma50 > -8:
        signals.append(("BUY CALL", "RSI oversold + holding MA50 support"))

    if trend == "UPTREND" and 0 < p_ma20 < 5 and rsi < 60 and not bearish_rev:
        signals.append(("BUY CALL", "Clean uptrend, not extended, no reversal signals"))

    if bullish_rev and rsi < 50 and trend != "DOWNTREND":
        signals.append(("BUY CALL", "Bullish reversal signals present"))

    # PUT signals
    if trend == "UPTREND" and rev_conviction == "HIGH":
        signals.append(("BUY PUT", f"Uptrend reversal - HIGH conviction [{rev_count} signals]"))

    if rsi > RSI_OVERBOUGHT and trend == "DOWNTREND" and p_ma50 < -5:
        signals.append(("BUY PUT", "RSI overbought in confirmed downtrend"))

    if ENABLE_REGIME_FILTER and market_regime == "BEARISH":
        if trend == "DOWNTREND" and bearish_rev and rev_conviction in ("MODERATE", "HIGH"):
            signals.append(("BUY PUT", f"Bearish regime + bearish reversals [{rev_count} signals]"))

    # De-dupe
    deduped_signals = []
    seen = set()
    for sig in signals:
        if sig not in seen:
            deduped_signals.append(sig)
            seen.add(sig)
    signals = deduped_signals

    # ──────────────────────────────────────────────────────────────
    # MULTI-TIMEFRAME FILTERING
    # ──────────────────────────────────────────────────────────────
    if signals and weekly_data and ENABLE_MTF:
        weekly_trend = weekly_data['trend']
        filtered_signals = []

        for direction, reason in signals:
            is_aligned, confidence_boost, mtf_desc = check_timeframe_alignment(
                trend, weekly_trend, direction
            )

            if REQUIRE_MTF_ALIGNMENT and not is_aligned:
                print(f"\n ⚠ FILTERED (MTF): {direction} - {mtf_desc}")
                continue

            filtered_signals.append((direction, reason))

            if confidence_boost != 0:
                print(f"\n {mtf_desc}")

        signals = filtered_signals

    # ──────────────────────────────────────────────────────────────
    # MARKET REGIME FILTERING
    # ──────────────────────────────────────────────────────────────
    signals, regime_notes = filter_signals_for_regime(
        signals, market_regime, trend, rsi, bullish_rev
    )

    if regime_notes:
        print(f"\n ── REGIME FILTER NOTES ──")
        for note in regime_notes:
            print(f" ⚠ {note}")

    # Calculate signal strength
    strength = signal_strength_score(signals, rsi, p_ma20, trend)

    put_signals = [s for s in signals if s[0] == "BUY PUT"]
    if put_signals and rev_conviction in ("MODERATE", "HIGH"):
        if strength == "moderate":
            strength = "strong"
        elif strength == "weak":
            strength = "moderate"

    if signals and weekly_data and ENABLE_MTF:
        for direction, _ in signals:
            _, confidence_boost, _ = check_timeframe_alignment(
                trend, weekly_data['trend'], direction
            )
            if confidence_boost > 0:
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break

    if signals and ENABLE_REGIME_FILTER:
        for direction, reason in signals:
            if market_regime == "BULLISH" and direction == "BUY CALL":
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break
            if market_regime == "BEARISH" and direction == "BUY PUT":
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"
                break

    print(f"\n ── SIGNALS [Conviction: {strength.upper()}] ──")
    if not signals:
        print(" No high-probability setup today")

        plot_analysis(symbol, hist, close, rsi_s, ma_s, ma_l,
                      reversal_signals, signals, price)

        if ENABLE_JOURNAL:
            journal_entry = create_journal_entry(
                symbol, price, rsi, trend, market_regime, p_ma20, p_ma50,
                reversal_signals, [], "none",
                expiry, dte, None, None
            )
            log_to_journal(journal_entry)

        return

    for direction, reason in signals:
        print(f" + {direction} — {reason}")

    if expiry:
        print(f" Target: {expiry} ({dte} DTE), near-ATM strike")

    # ──────────────────────────────────────────────────────────────
    # VISUALIZATION
    # ──────────────────────────────────────────────────────────────
    plot_analysis(symbol, hist, close, rsi_s, ma_s, ma_l,
                  reversal_signals, signals, price)

    if not expiry:
        print("\n No valid expiry found — cannot size position")
        return

    # ──────────────────────────────────────────────────────────────
    # OPTIONS CHAIN & POSITION SIZING
    # ──────────────────────────────────────────────────────────────
    atm_calls, atm_puts = get_atm_options(ticker, expiry, price)
    time_to_expiry = dte / 365.0 if dte else 0.123

    directions_in_order = []
    for d, _ in signals:
        if d not in directions_in_order:
            directions_in_order.append(d)

    for direction in directions_in_order:
        chain_df = atm_calls if direction == "BUY CALL" else atm_puts
        label = "CALL" if direction == "BUY CALL" else "PUT"
        option_type = "call" if direction == "BUY CALL" else "put"

        if chain_df.empty:
            print(f"\n ⚠ No liquid {label} options available")
            print(f"    (Need {MIN_OPTION_VOLUME}+ volume OR {MIN_OPTION_OI}+ open interest)")
            continue

        best = chain_df.iloc[0]
        option_ask = best.get('ask', 0)
        option_bid = best.get('bid', 0)
        option_iv = round(best.get('impliedVolatility', 0) * 100, 1)
        option_oi = int(best.get('openInterest', 0))
        option_volume = int(best.get('volume', 0)) if pd.notna(best.get('volume')) else 0
        spread_pct = best.get('spread_pct', 0)

        greeks = get_greeks_for_option(best, price, time_to_expiry, option_type)
        delta = greeks['delta']
        gamma = greeks['gamma']
        theta = greeks['theta']
        vega = greeks['vega']

        print(f"\n ── {label} CHAIN PREVIEW ({expiry}) ──")
        print(f" Strike     : ${best['strike']}")
        print(f" Bid/Ask    : ${option_bid:.2f} / ${option_ask:.2f} (spread: {spread_pct:.1f}%)")
        print(f" IV         : {option_iv}%")
        print(f" Volume     : {option_volume}")
        print(f" Open Int   : {option_oi}")

        print(f"\n ── GREEKS ──")
        if not np.isnan(delta):
            print(f" Delta      : {delta:.3f} (${abs(delta)*100:.0f} exposure per $1 stock move)")
        else:
            print(" Delta      : N/A")

        if not np.isnan(gamma):
            print(f" Gamma      : {gamma:.4f}")
        else:
            print(" Gamma      : N/A")

        if not np.isnan(theta):
            theta_per_day = theta * 100
            print(f" Theta      : {theta:.3f} (${theta_per_day:.2f}/day time decay)")
            if abs(theta_per_day) > option_ask * 100 * 0.05:
                print("            → ⚠ High time decay, need quick move")
        else:
            print(" Theta      : N/A")

        if not np.isnan(vega):
            vega_dollars = vega * 100
            print(f" Vega       : {vega:.3f} (${vega_dollars:.2f} per 1% IV change)")
            if vega_dollars > option_ask * 100 * 0.10:
                print("            → High IV sensitivity, watch for IV crush")
        else:
            print(" Vega       : N/A")

        liquidity_issues = []

        if option_volume < MIN_OPTION_VOLUME and option_oi < MIN_OPTION_OI:
            liquidity_issues.append(f"Low liquidity (vol: {option_volume}, OI: {option_oi})")

        if spread_pct > MAX_SPREAD_PCT:
            liquidity_issues.append(f"Wide spread ({spread_pct:.1f}% > {MAX_SPREAD_PCT}%)")

        if option_volume == 0:
            liquidity_issues.append("No volume today (stale pricing)")

        if liquidity_issues:
            print(f"\n ⚠ LIQUIDITY WARNINGS:")
            for issue in liquidity_issues:
                print(f"   • {issue}")
            print("   → Consider next expiry or different strike")

        if option_ask <= 0:
            print(" ⚠ No valid ask price — market may be closed")
            continue

        sizing = position_sizing(ACCOUNT_SIZE, option_ask, strength, iv_pct)

        if sizing:
            print_sizing(sizing, direction, option_ask)

            if not np.isnan(theta) and not np.isnan(delta) and sizing["contracts"] > 0:
                total_theta = theta * 100 * sizing["contracts"]
                total_delta_exposure = abs(delta) * price * sizing["contracts"]

                print(f"\n ── POSITION GREEKS SUMMARY ──")
                print(f" Total Delta Exposure: ${total_delta_exposure:,.0f}")
                print(f" Daily Theta Cost    : ${abs(total_theta):.2f}/day")

                if abs(delta) > 0.01:
                    breakeven_move = abs(total_theta / (abs(delta) * sizing['contracts']))
                    print(f" Break-even Move     : ${breakeven_move:.2f}/day in stock")

                days_to_expiry = dte if dte else 45
                total_decay = abs(total_theta) * days_to_expiry
                if total_decay > sizing["total_cost"] * 0.5:
                    print(f" ⚠ Time decay will erode {total_decay/sizing['total_cost']*100:.0f}% of premium by expiry")

            if ENABLE_JOURNAL:
                option_data = {
                    'strike': best['strike'],
                    'ask': option_ask,
                    'iv': option_iv,
                    'delta': delta if not np.isnan(delta) else 0,
                    'theta': theta if not np.isnan(theta) else 0
                }

                journal_entry = create_journal_entry(
                    symbol, price, rsi, trend, market_regime, p_ma20, p_ma50,
                    reversal_signals, signals, strength,
                    expiry, dte, option_data, sizing
                )
                log_to_journal(journal_entry)
                print(f"\n ✓ Logged to journal: {JOURNAL_PATH}")

        else:
            print(" ⚠ Unable to calculate position size (invalid inputs)")


print("✓ Main scanner function defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Main scanner function defined
  Notebook Version: v2.5
  Cell Version: v2.5


## Text Cell 8A: Backtesting Module

### Purpose
Backtesting validates your strategy by simulating trades on historical data. It shows how the scanner would have performed in the past, helping you understand expected win rate, average returns, and drawdowns before risking real capital.

### How It Works
1. **Walk-forward analysis**: Steps through historical data day-by-day
2. **Signal detection**: Runs scanner logic at each point in time (no future data)
3. **Trade simulation**: Enters positions when signals trigger, exits after hold period or at stop/target
4. **P&L calculation**: Estimates option gains/losses based on stock price moves and time decay
5. **Performance metrics**: Calculates win rate, avg return, max drawdown, profit factor

### Simplified Option P&L Model
Since historical option prices aren't available, we estimate P&L using:
- **Delta-based approximation**: Option moves ~$50 per $1 stock move (for 0.50 delta)
- **Theta decay**: Loses value daily based on time to expiration
- **Entry slippage**: 5% worse fill than theoretical price
- **Exit rules**: -50% stop loss, +100% profit target, or exit after 21 days

### Configuration
- `BACKTEST_MODE = True/False` - Toggle backtest vs live scanning
- `BACKTEST_START_DATE` / `BACKTEST_END_DATE` - Date range to test
- `BACKTEST_HOLD_DAYS = 21` - How long to hold positions
- `BACKTEST_SLIPPAGE = 0.05` - 5% execution slippage
- `BACKTEST_COMMISSION = 1.00` - $1 per contract fees

### Limitations
- **No actual option prices**: Uses simplified delta/theta model
- **No IV changes**: Doesn't model volatility crush or expansion
- **Assumes liquidity**: Can't backtest liquidity filters accurately
- **Survivorship bias**: Only tests current tickers (no delisted stocks)

### Output
- **Console**: Summary stats (total trades, win rate, avg return, etc.)
- **CSV**: Trade-by-trade results exported to `backtest_results.csv`

### Best Practice
Run backtest on 1-2 years of data, then paper trade 1-3 months before going live with real money.

In [22]:
# Code Cell 8A: Backtesting Framework
# Notebook Version: v2.5
# Cell Version: v2.5

from datetime import timedelta
import yfinance as yf
import pandas as pd
import numpy as np

def estimate_option_pnl(direction, entry_price_stock, exit_price_stock,
                       days_held, entry_premium, delta=0.5):
    """
    Estimate option P&L based on stock price movement and time decay.
    """
    stock_move = exit_price_stock - entry_price_stock

    if direction == "CALL":
        option_price_change = stock_move * delta
    else:
        option_price_change = -stock_move * delta

    weeks_held = days_held / 7.0
    theta_decay_pct = -0.02 * weeks_held
    theta_decay_dollars = entry_premium * theta_decay_pct

    net_change = option_price_change + theta_decay_dollars
    pnl_pct = net_change / entry_premium

    return pnl_pct


def run_backtest_on_ticker(symbol):
    """
    Run backtest on single ticker over configured date range.
    Prevents overlapping trades on the same ticker.
    """
    print(f"\n{'='*62}")
    print(f" BACKTESTING: {symbol}")
    print(f"{'='*62}")

    trades = []

    try:
        ticker = yf.Ticker(symbol)
        hist = ticker.history(start=BACKTEST_START_DATE, end=BACKTEST_END_DATE)

        benchmark_hist = None
        if ENABLE_REGIME_FILTER:
            benchmark_start = (
                pd.to_datetime(BACKTEST_START_DATE) - pd.Timedelta(days=REGIME_WARMUP_DAYS)
            ).strftime('%Y-%m-%d')

            benchmark_hist = yf.Ticker(REGIME_SYMBOL).history(
                start=benchmark_start,
                end=BACKTEST_END_DATE
            )

        if hist.empty or len(hist) < MA_LONG + BACKTEST_HOLD_DAYS + 10:
            print(" ⚠ Insufficient data for backtest period")
            return trades

        i = MA_LONG + 5
        while i < len(hist) - BACKTEST_HOLD_DAYS:
            current_date = hist.index[i]

            hist_slice = hist.iloc[:i+1]
            close = hist_slice['Close']
            volume = hist_slice['Volume']
            price = round(close.iloc[-1], 2)

            rsi_s = compute_rsi(close)
            rsi = round(rsi_s.iloc[-1], 1)

            if np.isnan(rsi):
                i += 1
                continue

            ma_s = close.rolling(MA_SHORT).mean()
            ma_l = close.rolling(MA_LONG).mean()
            ma_s_current = ma_s.iloc[-1]
            ma_l_current = ma_l.iloc[-1]

            trend = get_trend(price, ma_s_current, ma_l_current)

            p_ma20 = round((price - ma_s_current) / ma_s_current * 100, 2)
            p_ma50 = round((price - ma_l_current) / ma_l_current * 100, 2)

            # Historical market regime at trade date
            market_regime = "UNKNOWN"
            if ENABLE_REGIME_FILTER and benchmark_hist is not None and not benchmark_hist.empty:
                benchmark_slice = benchmark_hist.loc[:current_date]
                if not benchmark_slice.empty:
                    regime_data = get_regime_from_history(
                        benchmark_slice['Close'],
                        REGIME_MA_SHORT,
                        REGIME_MA_LONG
                    )
                    if regime_data:
                        market_regime = regime_data['regime']

            # Detect reversals
            divergences = detect_rsi_divergence(close, rsi_s)
            vol_signals = detect_volume_signals(close, volume)
            climax, c_det = detect_climax(close, volume)
            failed_bo, f_det = detect_failed_breakout(close)
            ma_break, m_det = detect_ma_breakdown(close, ma_s, "MA20")

            reversal_signals = []

            if "BEARISH_DIVERGENCE" in divergences:
                reversal_signals.append(("BEARISH", "RSI bearish divergence"))
            if "BULLISH_DIVERGENCE" in divergences:
                reversal_signals.append(("BULLISH", "RSI bullish divergence"))
            if "DISTRIBUTION" in vol_signals:
                reversal_signals.append(("BEARISH", "Distribution"))
            if "SELLING_PRESSURE" in vol_signals:
                reversal_signals.append(("BEARISH", "Selling pressure"))
            if "WEAK_RALLY" in vol_signals:
                reversal_signals.append(("BEARISH", "Weak rally"))
            if "ACCUMULATION" in vol_signals:
                reversal_signals.append(("BULLISH", "Accumulation"))
            if climax:
                reversal_signals.append(("BEARISH", "Climax top"))
            if failed_bo:
                reversal_signals.append(("BEARISH", "Failed breakout"))
            if ma_break:
                reversal_signals.append(("BEARISH", "MA breakdown"))

            bearish_rev = [r for r in reversal_signals if r[0] == "BEARISH"]
            bullish_rev = [r for r in reversal_signals if r[0] == "BULLISH"]
            rev_conviction, rev_count = score_reversal(bearish_rev)

            # Signal generation
            signals = []

            # CALL signals
            if rsi < RSI_OVERSOLD and trend in ("UPTREND", "MIXED") and p_ma50 > -8:
                signals.append(("BUY CALL", "RSI oversold + MA50 support"))

            if trend == "UPTREND" and 0 < p_ma20 < 5 and rsi < 60 and not bearish_rev:
                signals.append(("BUY CALL", "Clean uptrend"))

            if bullish_rev and rsi < 50 and trend != "DOWNTREND":
                signals.append(("BUY CALL", "Bullish reversal"))

            # PUT signals
            if trend == "UPTREND" and rev_conviction == "HIGH":
                signals.append(("BUY PUT", "Uptrend reversal - HIGH conviction"))

            if rsi > RSI_OVERBOUGHT and trend == "DOWNTREND" and p_ma50 < -5:
                signals.append(("BUY PUT", "Overbought in downtrend"))

            if ENABLE_REGIME_FILTER and market_regime == "BEARISH":
                if trend == "DOWNTREND" and bearish_rev and rev_conviction in ("MODERATE", "HIGH"):
                    signals.append(("BUY PUT", "Bearish regime + bearish reversals"))

            # De-dupe
            deduped_signals = []
            seen = set()
            for sig in signals:
                if sig not in seen:
                    deduped_signals.append(sig)
                    seen.add(sig)
            signals = deduped_signals

            # Apply regime filter
            signals, regime_notes = filter_signals_for_regime(
                signals, market_regime, trend, rsi, bullish_rev
            )

            if not signals:
                i += 1
                continue

            strength = signal_strength_score(signals, rsi, p_ma20, trend)

            put_signals = [s for s in signals if s[0] == "BUY PUT"]
            if put_signals and rev_conviction in ("MODERATE", "HIGH"):
                if strength == "moderate":
                    strength = "strong"
                elif strength == "weak":
                    strength = "moderate"

            trade_taken = False

            for direction, reason in signals:
                estimated_premium = price * 0.03
                entry_premium = estimated_premium * (1 + BACKTEST_SLIPPAGE)

                tier_map = {
                    "strong": RISK_PCT_MAX,
                    "moderate": (RISK_PCT_MIN + RISK_PCT_MAX) / 2,
                    "weak": RISK_PCT_MIN
                }
                risk_pct = tier_map.get(strength, RISK_PCT_MIN)
                risk_budget = ACCOUNT_SIZE * risk_pct
                contracts = max(int(risk_budget / (entry_premium * 100)), 1)

                entry_cost = contracts * entry_premium * 100

                exit_idx = min(i + BACKTEST_HOLD_DAYS, len(hist) - 1)
                exit_date = hist.index[exit_idx]
                exit_price_stock = hist['Close'].iloc[exit_idx]
                days_held = (exit_date - current_date).days

                pnl_pct = estimate_option_pnl(
                    direction.replace("BUY ", ""),
                    price,
                    exit_price_stock,
                    days_held,
                    entry_premium
                )

                if pnl_pct <= -STOP_LOSS_PCT:
                    pnl_pct = -STOP_LOSS_PCT
                    exit_reason = "STOP LOSS"
                elif pnl_pct >= PROFIT_TARGET_PCT:
                    pnl_pct = PROFIT_TARGET_PCT
                    exit_reason = "PROFIT TARGET"
                else:
                    exit_reason = f"TIME ({days_held}d)"

                exit_premium = entry_premium * (1 + pnl_pct)
                exit_cost = contracts * exit_premium * 100
                pnl_dollars = exit_cost - entry_cost - (contracts * BACKTEST_COMMISSION)

                trade = {
                    'entry_date': current_date.strftime('%Y-%m-%d'),
                    'exit_date': exit_date.strftime('%Y-%m-%d'),
                    'symbol': symbol,
                    'direction': direction,
                    'reason': reason,
                    'conviction': strength,
                    'market_regime': market_regime,
                    'entry_stock_price': price,
                    'exit_stock_price': round(exit_price_stock, 2),
                    'entry_premium': round(entry_premium, 2),
                    'exit_premium': round(exit_premium, 2),
                    'contracts': contracts,
                    'entry_cost': round(entry_cost, 2),
                    'exit_cost': round(exit_cost, 2),
                    'pnl_dollars': round(pnl_dollars, 2),
                    'pnl_pct': round(pnl_pct * 100, 1),
                    'days_held': days_held,
                    'exit_reason': exit_reason,
                    'rsi': rsi,
                    'trend': trend
                }

                trades.append(trade)
                i = exit_idx + 1
                trade_taken = True
                break

            if not trade_taken:
                i += 1

    except Exception as e:
        print(f" ⚠ Backtest error: {e}")

    return trades


def analyze_backtest_results(all_trades):
    """
    Calculate performance metrics from backtest trades.
    """
    if not all_trades:
        return None

    df = pd.DataFrame(all_trades).copy()
    df['entry_date'] = pd.to_datetime(df['entry_date'])
    df = df.sort_values('entry_date').reset_index(drop=True)

    total_trades = len(df)
    winning_trades = len(df[df['pnl_dollars'] > 0])
    losing_trades = len(df[df['pnl_dollars'] <= 0])

    win_rate = (winning_trades / total_trades * 100) if total_trades > 0 else 0

    total_pnl = df['pnl_dollars'].sum()
    avg_win = df[df['pnl_dollars'] > 0]['pnl_dollars'].mean() if winning_trades > 0 else 0
    avg_loss = df[df['pnl_dollars'] <= 0]['pnl_dollars'].mean() if losing_trades > 0 else 0

    largest_win = df['pnl_dollars'].max()
    largest_loss = df['pnl_dollars'].min()

    df['cumulative_pnl'] = df['pnl_dollars'].cumsum()
    df['running_max'] = df['cumulative_pnl'].cummax()
    df['drawdown'] = df['cumulative_pnl'] - df['running_max']
    max_drawdown = df['drawdown'].min()

    gross_profit = df[df['pnl_dollars'] > 0]['pnl_dollars'].sum()
    gross_loss = abs(df[df['pnl_dollars'] <= 0]['pnl_dollars'].sum())
    profit_factor = (gross_profit / gross_loss) if gross_loss > 0 else float('inf')

    return {
        'total_trades': total_trades,
        'winning_trades': winning_trades,
        'losing_trades': losing_trades,
        'win_rate': round(win_rate, 1),
        'total_pnl': round(total_pnl, 2),
        'avg_win': round(avg_win, 2),
        'avg_loss': round(avg_loss, 2),
        'largest_win': round(largest_win, 2),
        'largest_loss': round(largest_loss, 2),
        'max_drawdown': round(max_drawdown, 2),
        'profit_factor': round(profit_factor, 2) if profit_factor != float('inf') else 'INF',
        'avg_pnl_per_trade': round(total_pnl / total_trades, 2) if total_trades > 0 else 0
    }


print("✓ Backtesting functions defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Backtesting functions defined
  Notebook Version: v2.5
  Cell Version: v2.5


## Text Cell 8: Run Scanner

### Execution
This cell runs the complete scanner across all configured tickers. For each symbol, it will display:
- Current market data and technical levels
- Detected reversal patterns with conviction scores
- Directional trade signals (calls/puts) with rationale
- Options chain data for recommended strikes
- Position sizing calculations with risk metrics

### Customization
To modify the watchlist, edit the `TICKERS` list in Code Cell 2. The scanner processes each ticker sequentially and displays results in real-time.

In [23]:
# Code Cell 8: Execute Scanner / Backtest
# Notebook Version: v2.5
# Cell Version: v2.5

if __name__ == "__main__":
    try:
        from datetime import datetime, timezone as dt_timezone
        from pytz import timezone as pytz_timezone

        utc_now = datetime.now(dt_timezone.utc)
        tz = pytz_timezone(TIMEZONE)
        local_time = utc_now.astimezone(tz)
        timestamp_display = local_time.strftime('%Y-%m-%d %H:%M %Z')
    except Exception:
        timestamp_display = datetime.now().strftime('%Y-%m-%d %H:%M')

    # ──────────────────────────────────────────────────────────────
    # BACKTEST MODE
    # ──────────────────────────────────────────────────────────────
    if BACKTEST_MODE:
        print("\n" + "█"*62)
        print(" BACKTEST MODE — Historical Performance Analysis")
        print(" Notebook Version: v2.5")
        print(" Execution Cell Version: v2.5")
        print(f" Period: {BACKTEST_START_DATE} to {BACKTEST_END_DATE}")
        print(f" Hold Period: {BACKTEST_HOLD_DAYS} trading days")
        print(f" Tickers: {', '.join(TICKERS)}")
        print(f" Regime Filter: {'Enabled' if ENABLE_REGIME_FILTER else 'Disabled'} ({REGIME_SYMBOL})")
        print(f" Regime Warm-Up Days: {REGIME_WARMUP_DAYS}")
        print("█"*62)

        all_trades = []

        for t in TICKERS:
            trades = run_backtest_on_ticker(t)
            all_trades.extend(trades)
            print(f" {t}: {len(trades)} trades generated")

        if all_trades:
            df_trades = pd.DataFrame(all_trades)
            df_trades.to_csv(BACKTEST_RESULTS_PATH, index=False)
            print(f"\n✓ Trade results exported to: {BACKTEST_RESULTS_PATH}")

            metrics = analyze_backtest_results(all_trades)

            print(f"\n{'='*62}")
            print(" BACKTEST RESULTS")
            print(f"{'='*62}")
            print(f" Total Trades      : {metrics['total_trades']}")
            print(f" Winning Trades    : {metrics['winning_trades']}")
            print(f" Losing Trades     : {metrics['losing_trades']}")
            print(f" Win Rate          : {metrics['win_rate']}%")
            print(f" Total P&L         : ${metrics['total_pnl']:,.2f}")
            print(f" Avg Win           : ${metrics['avg_win']:,.2f}")
            print(f" Avg Loss          : ${metrics['avg_loss']:,.2f}")
            print(f" Avg P&L/Trade     : ${metrics['avg_pnl_per_trade']:,.2f}")
            print(f" Largest Win       : ${metrics['largest_win']:,.2f}")
            print(f" Largest Loss      : ${metrics['largest_loss']:,.2f}")
            print(f" Max Drawdown      : ${metrics['max_drawdown']:,.2f}")
            print(f" Profit Factor     : {metrics['profit_factor']}")
            print(f"{'='*62}")

            if metrics['win_rate'] >= 60 and (metrics['profit_factor'] == 'INF' or float(metrics['profit_factor']) >= 2.0):
                rating = "EXCELLENT"
            elif metrics['win_rate'] >= 50 and (metrics['profit_factor'] == 'INF' or float(metrics['profit_factor']) >= 1.5):
                rating = "GOOD"
            elif metrics['win_rate'] >= 40 and (metrics['profit_factor'] == 'INF' or float(metrics['profit_factor']) >= 1.0):
                rating = "ACCEPTABLE"
            else:
                rating = "NEEDS IMPROVEMENT"

            print(f"\n Strategy Rating: {rating}")
            print(f" (Win Rate: {metrics['win_rate']}%, Profit Factor: {metrics['profit_factor']})")

            print_backtest_diagnostics(all_trades)

        else:
            print("\n⚠ No trades generated during backtest period")

    # ──────────────────────────────────────────────────────────────
    # LIVE SCAN MODE
    # ──────────────────────────────────────────────────────────────
    else:
        print("\n" + "█"*62)
        print(" OPTIONS SCANNER — Directional Buys (45+ DTE)")
        print(" Notebook Version: v2.5")
        print(" Execution Cell Version: v2.5")
        print(f" Account: ${ACCOUNT_SIZE:,.2f} | Risk: {RISK_PCT_MIN*100:.0f}%–{RISK_PCT_MAX*100:.0f}% per trade")
        print(f" Regime Filter: {'Enabled' if ENABLE_REGIME_FILTER else 'Disabled'} ({REGIME_SYMBOL})")
        print(f" Run date: {timestamp_display}")
        print("█"*62)

        for t in TICKERS:
            evaluate_ticker(t)

        print(f"\n{'='*62}")
        print(" SCAN COMPLETE")
        print(f"{'='*62}\n")


██████████████████████████████████████████████████████████████
 BACKTEST MODE — Historical Performance Analysis
 Notebook Version: v2.5
 Execution Cell Version: v2.5
 Period: 2018-01-01 to 2024-12-31
 Hold Period: 17 trading days
 Tickers: GOOGL, AAPL, MSFT
 Regime Filter: Enabled (SPY)
 Regime Warm-Up Days: 400
██████████████████████████████████████████████████████████████

 BACKTESTING: GOOGL
 GOOGL: 42 trades generated

 BACKTESTING: AAPL
 AAPL: 40 trades generated

 BACKTESTING: MSFT
 MSFT: 50 trades generated

✓ Trade results exported to: backtest_results.csv

 BACKTEST RESULTS
 Total Trades      : 132
 Winning Trades    : 73
 Losing Trades     : 59
 Win Rate          : 55.3%
 Total P&L         : $10,184.68
 Avg Win           : $261.10
 Avg Loss          : $-150.43
 Avg P&L/Trade     : $77.16
 Largest Win       : $987.35
 Largest Loss      : $-494.37
 Max Drawdown      : $-2,470.67
 Profit Factor     : 2.15

 Strategy Rating: GOOD
 (Win Rate: 55.3%, Profit Factor: 2.15)

 BACKTES

## Text Cell 8B: Backtest Diagnostics

### Purpose
Raw summary metrics are useful, but they do not show where the edge is actually coming from. This diagnostic layer breaks backtest performance down by:

- **Year**: to evaluate market-regime sensitivity
- **Symbol**: to identify ticker bias
- **Setup Reason**: to identify strongest and weakest entry logic
- **Exit Reason**: to see whether trades are winning via target, timing, or something else
- **Market Regime**: to evaluate whether the filter is improving behavior in bullish, mixed, and bearish environments
- **Setup by Regime**: to identify which specific entries fail outside bullish conditions

### Why It Matters
A strategy can appear profitable overall while still depending too heavily on:
- one market regime
- one symbol
- one setup type

v2.5 places special emphasis on **setup performance inside MIXED regimes**, because that is where v2.4 still showed meaningful weakness.

In [25]:
# Code Cell 8B: Backtest Diagnostics
# Notebook Version: v2.5
# Cell Version: v2.5

def _group_performance_table(df, group_cols):
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    rows = []
    for key, g in df.groupby(group_cols):
        if not isinstance(key, tuple):
            key = (key,)

        row = {col: val for col, val in zip(group_cols, key)}
        trades = len(g)
        wins = int((g['pnl_dollars'] > 0).sum())
        losses = int((g['pnl_dollars'] <= 0).sum())
        win_rate = round((wins / trades) * 100, 1) if trades > 0 else 0
        total_pnl = round(g['pnl_dollars'].sum(), 2)
        avg_pnl = round(g['pnl_dollars'].mean(), 2)

        row.update({
            'trades': trades,
            'wins': wins,
            'losses': losses,
            'win_rate_pct': win_rate,
            'total_pnl': total_pnl,
            'avg_pnl': avg_pnl
        })
        rows.append(row)

    return pd.DataFrame(rows)


def print_backtest_diagnostics(all_trades):
    """
    Print diagnostic tables for backtest review.
    """
    if not all_trades:
        print("\n⚠ No trades available for diagnostics")
        return

    df = pd.DataFrame(all_trades).copy()
    df['entry_date'] = pd.to_datetime(df['entry_date'])
    df['entry_year'] = df['entry_date'].dt.year

    print(f"\n{'='*62}")
    print(" BACKTEST DIAGNOSTICS")
    print(f"{'='*62}")

    print("\n--- By Year ---")
    by_year = _group_performance_table(df, 'entry_year').sort_values('entry_year')
    print(by_year.to_string(index=False))

    print("\n--- By Symbol ---")
    by_symbol = _group_performance_table(df, 'symbol').sort_values('total_pnl', ascending=False)
    print(by_symbol.to_string(index=False))

    print("\n--- By Setup Reason ---")
    by_reason = _group_performance_table(df, 'reason').sort_values('total_pnl', ascending=False)
    print(by_reason.to_string(index=False))

    print("\n--- By Exit Reason ---")
    by_exit = _group_performance_table(df, 'exit_reason').sort_values('trades', ascending=False)
    print(by_exit.to_string(index=False))

    if 'market_regime' in df.columns:
        print("\n--- By Market Regime ---")
        by_regime = _group_performance_table(df, 'market_regime').sort_values('total_pnl', ascending=False)
        print(by_regime.to_string(index=False))

        print("\n--- By Setup Reason AND Market Regime ---")
        by_reason_regime = _group_performance_table(df, ['market_regime', 'reason']).sort_values(
            ['market_regime', 'total_pnl'],
            ascending=[True, False]
        )
        print(by_reason_regime.to_string(index=False))


print("✓ Backtest diagnostics defined")
print("  Notebook Version: v2.5")
print("  Cell Version: v2.5")

✓ Backtest diagnostics defined
  Notebook Version: v2.5
  Cell Version: v2.5


## Final Version Review

**Notebook Version**: v2.5  
**Focus of This Version**: Mixed-Regime Tightening  
**Backtest Window**: 2018-01-01 to 2024-12-31  
**Tickers Tested**: GOOGL, AAPL, MSFT  
**Backtest Summary**: 132 trades, 73 winners, 59 losers, 55.3% win rate

### The Good ✅
- Benchmark warm-up history worked as intended and reduced early regime-labeling issues.
- Mixed-regime tightening clearly reduced trade count, showing that the filter is active and not just cosmetic.
- MSFT remains the strongest symbol and continues to support the core strategy edge.
- The strategy is still functional and was not broken by the v2.5 changes.
- Selective bearish-regime countertrend CALL exceptions did not appear to be the main problem.

### The Bad ⚠️
- v2.5 does not appear to have improved overall performance enough to justify the tighter filtering.
- Filtering `Clean uptrend` CALLs in MIXED regimes reduced trades, but did not solve the broader mixed-regime weakness.
- The strategy still behaves primarily as a bullish long-call system.
- PUT participation remains too limited and too weak to offset bullish bias.

### The Ugly 🚨
- The evidence suggests that MIXED conditions remain structurally difficult, not just for `Clean uptrend` CALLs, but for other weaker bullish setups as well.
- v2.5 may have filtered out some acceptable trades while still leaving other weak mixed-regime trades in place.
- AAPL and GOOGL remain much less reliable than MSFT, which raises ongoing concentration and symbol-bias concerns.
- The strategy is still not regime-robust enough to treat mixed-market conditions confidently.

### Ideas for Future Versions 💡
- Tighten **all weak CALL setups** in MIXED regimes, not just `Clean uptrend`.
- Consider requiring stronger confirmation in MIXED conditions, such as higher conviction, stronger bullish reversals, or non-bearish weekly alignment.
- Compare setup performance inside MIXED regimes to identify whether `RSI oversold + MA50 support` or `Bullish reversal` should also be restricted.
- Test `QQQ` as a regime benchmark to see whether it fits a tech-heavy strategy better than `SPY`.
- Delay any major PUT expansion or spread work until the mixed-regime logic is cleaner.